# EconoCausal — DoWhy Refutation Audit (minimal safe fix)

This version reuses the processed files already in the `refutation test` folder. It audits promotion-code counts and key coverage without requiring the 663 MB raw promotion file.

**Important:** `display_code_count` and `mailer_code_count` are counts, not verified exposure labels. This notebook therefore does not estimate causal effects or run DoWhy refuters from these counts. It exports a transparent audit and explains what is still missing.


## 1. Imports and paths

This cell only imports packages and sets the project folder. It does not search for files yet.

In [1]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import duckdb
import dowhy
from dowhy import CausalModel

PROJECT_DIR = Path.cwd()
SEARCH_ROOTS = [PROJECT_DIR, PROJECT_DIR.parent]
SKIP_DIRS = {".git", ".venv", "venv", "env", "__pycache__", "node_modules", ".ipynb_checkpoints"}
OUTPUT_DIR = PROJECT_DIR / "causal_audit_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR.resolve())
print("Python packages imported successfully.")
print("pandas:", pd.__version__, "| NumPy:", np.__version__,
      "| DuckDB:", duckdb.__version__, "| DoWhy:", dowhy.__version__)

d:\EconoCausal_Env\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project folder: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test
Python packages imported successfully.
pandas: 3.0.6 | NumPy: 2.4.6 | DuckDB: 2.0.0 | DoWhy: 0.14


## 2. Find the existing processed input files

The notebook checks its current folder and a few nearby project folders. If it cannot find a file, set its path explicitly below.


In [2]:
# Set these paths explicitly if the notebook is not inside the `refutation test` folder.
TRANSACTION_CANDIDATES = [
    Path.cwd() / "transaction_weekly_final.csv",
    Path.cwd().parent / "transaction_weekly_final.csv",
]
PROMOTION_CANDIDATES = [
    Path.cwd() / "promotion_features.csv",
    Path.cwd().parent / "promotion_features.csv",
]

def first_existing(candidates):
    return next((p.resolve() for p in candidates if p.is_file()), None)

TRANSACTION_PATH = first_existing(TRANSACTION_CANDIDATES)
PROMOTION_PATH = first_existing(PROMOTION_CANDIDATES)
print("Transaction file:", TRANSACTION_PATH or "NOT FOUND")
print("Processed promotion features:", PROMOTION_PATH or "NOT FOUND")


Transaction file: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\transaction_weekly_final.csv
Processed promotion features: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\promotion_features.csv


## 3. Validate paths and schemas

This audit accepts the processed promotion feature file, but does not treat its counts as actual exposure.


In [3]:
if TRANSACTION_PATH is None:
    raise FileNotFoundError("Could not find transaction_weekly_final.csv. Set TRANSACTION_PATH in Cell 4 to its full path.")
if PROMOTION_PATH is None:
    raise FileNotFoundError("Could not find promotion_features.csv. Set PROMOTION_PATH in Cell 4 to its full path.")

print("Transaction:", TRANSACTION_PATH)
print("Promotion features:", PROMOTION_PATH)


Transaction: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\transaction_weekly_final.csv
Promotion features: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\promotion_features.csv


## 4. Audit the existing files with DuckDB

DuckDB scans the CSV files without loading all 663 MB of promotion features into a pandas DataFrame.


In [4]:
KEYS = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
OUTCOME = "TOTAL_QUANTITY"

con = duckdb.connect(database=":memory:")

def describe_csv(path):
    return con.execute("DESCRIBE SELECT * FROM read_csv_auto(?, header=true, sample_size=20000)", [str(path)]).df()["column_name"].tolist()

tx_columns = describe_csv(TRANSACTION_PATH)
promo_columns = describe_csv(PROMOTION_PATH)
print("Transaction columns:", tx_columns)
print("Promotion-feature columns:", promo_columns)

missing_tx = sorted(set(KEYS + [OUTCOME]) - set(tx_columns))
missing_promo = sorted(set(KEYS + ["display_code_count", "mailer_code_count"]) - set(promo_columns))
if missing_tx:
    raise ValueError(f"Transaction file is missing required columns: {missing_tx}")
if missing_promo:
    raise ValueError(f"Promotion feature file is missing required columns: {missing_promo}. This notebook expects the processed count-only file.")


Transaction columns: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'TOTAL_QUANTITY', 'TOTAL_SALES', 'TOTAL_RETAIL_DISC', 'TOTAL_COUPON_DISC', 'TOTAL_COUPON_MATCH_DISC', 'TRANSACTION_ROWS', 'UNIQUE_BASKETS', 'ZERO_QUANTITY_ROWS', 'ZERO_SALES_ROWS', 'EXTREME_QUANTITY_ROWS', 'PRICE_PROXY', 'ZERO_OR_NEGATIVE_QUANTITY_FLAG', 'ZERO_SALES_FLAG', 'MISSING_PRICE_PROXY_FLAG', 'EXTREME_QUANTITY_FLAG', 'NONPOSITIVE_PRICE_PROXY_FLAG']
Promotion-feature columns: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display_code_count', 'mailer_code_count']


## 5. Check key uniqueness and match coverage

The join coverage is descriptive only. Unmatched transactions remain unknown; they are not classified as untreated.


In [5]:
tx_sql = "read_csv_auto(?, header=true, sample_size=20000)"
promo_sql = "read_csv_auto(?, header=true, sample_size=20000)"

# Cast keys to VARCHAR consistently before comparing them.
tx_key_expr = ", ".join([f"TRIM(CAST({k} AS VARCHAR)) AS {k}" for k in KEYS])
promo_key_expr = ", ".join([f"TRIM(CAST({k} AS VARCHAR)) AS {k}" for k in KEYS])

coverage = con.execute(f"""
WITH tx AS (
  SELECT DISTINCT {tx_key_expr} FROM {tx_sql}
  WHERE PRODUCT_ID IS NOT NULL AND STORE_ID IS NOT NULL AND WEEK_NO IS NOT NULL
), promo AS (
  SELECT DISTINCT {promo_key_expr} FROM {promo_sql}
  WHERE PRODUCT_ID IS NOT NULL AND STORE_ID IS NOT NULL AND WEEK_NO IS NOT NULL
), counts AS (
  SELECT (SELECT COUNT(*) FROM tx) AS transaction_unique_keys,
         (SELECT COUNT(*) FROM promo) AS promotion_unique_keys,
         (SELECT COUNT(*) FROM tx JOIN promo USING (PRODUCT_ID, STORE_ID, WEEK_NO)) AS matched_keys
)
SELECT *, matched_keys * 1.0 / NULLIF(transaction_unique_keys, 0) AS match_rate,
       FALSE AS unmatched_assumed_untreated
FROM counts
""", [str(TRANSACTION_PATH), str(PROMOTION_PATH)]).df()
display(coverage)
coverage.to_csv(OUTPUT_DIR / "match_coverage.csv", index=False)
print("Saved match_coverage.csv")


,transaction_unique_keys,promotion_unique_keys,matched_keys,match_rate,unmatched_assumed_untreated
0,2370784,36771279,483320,0.203865,False


Saved match_coverage.csv


## 6. Inspect the code-count distributions

These are counts of codes per product/store/week, not raw exposure categories. A count of 1 does not mean exposed, and a count of 2 does not necessarily mean a stronger treatment.


In [6]:
count_diagnostics = con.execute(f"""
SELECT 'display_code_count' AS feature, CAST(display_code_count AS VARCHAR) AS value, COUNT(*) AS rows
FROM {promo_sql} GROUP BY 1, 2
UNION ALL
SELECT 'mailer_code_count' AS feature, CAST(mailer_code_count AS VARCHAR) AS value, COUNT(*) AS rows
FROM {promo_sql} GROUP BY 1, 2
ORDER BY feature, value
""", [str(PROMOTION_PATH), str(PROMOTION_PATH)]).df()
display(count_diagnostics)
count_diagnostics.to_csv(OUTPUT_DIR / "promotion_code_count_diagnostics.csv", index=False)
print("Saved promotion_code_count_diagnostics.csv")


,feature,value,rows
0,display_code_count,1,36756071
1,display_code_count,2,15208
2,mailer_code_count,1,36771174
3,mailer_code_count,2,105


Saved promotion_code_count_diagnostics.csv


## 7. Treatment readiness decision

The raw `display` and `mailer` exposure values are not present in `promotion_features.csv`. Therefore this processed file alone cannot establish the treatment definitions required for a defensible causal estimate.


In [7]:
MIN_PER_GROUP = 200
MIN_PREVALENCE = 0.01

readiness = pd.DataFrame([
    {"treatment": "display", "status": "BLOCKED", "reason": "Only display_code_count is available; verified exposure labels are absent."},
    {"treatment": "mailer", "status": "BLOCKED", "reason": "Only mailer_code_count is available; verified exposure labels are absent."},
])
display(readiness)
readiness.to_csv(OUTPUT_DIR / "treatment_readiness.csv", index=False)


,treatment,status,reason
0,display,BLOCKED,Only display_code_count is available; verified...
1,mailer,BLOCKED,Only mailer_code_count is available; verified ...


## 8. DoWhy estimates and refutation tests — intentionally not run

Do not set count thresholds just to force a model to run. Estimates and refutation tests remain blocked until treatment coding is confirmed from raw promotion values or authoritative documentation and treatment variation is rechecked on the matched analysis sample.


In [8]:
summary_df = pd.DataFrame([
    {"treatment": "display", "status": "NOT_ESTIMATED", "reason": "Processed code counts are not verified exposure labels."},
    {"treatment": "mailer", "status": "NOT_ESTIMATED", "reason": "Processed code counts are not verified exposure labels."},
])
refutation_df = pd.DataFrame(columns=["treatment", "method", "status", "original_effect", "new_effect", "details"])
display(summary_df)
display(refutation_df)
summary_df.to_csv(OUTPUT_DIR / "dowhy_estimate_summary.csv", index=False)
refutation_df.to_csv(OUTPUT_DIR / "dowhy_refutation_results.csv", index=False)
print("Saved safe placeholder outputs in:", OUTPUT_DIR.resolve())


,treatment,status,reason
0,display,NOT_ESTIMATED,Processed code counts are not verified exposur...
1,mailer,NOT_ESTIMATED,Processed code counts are not verified exposur...


,treatment,method,status,original_effect,new_effect,details


Saved safe placeholder outputs in: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\causal_audit_outputs


## 9. Export the audit summary


In [9]:
audit = {
    "project_dir": str(Path.cwd().resolve()),
    "transaction_path": str(TRANSACTION_PATH),
    "promotion_features_path": str(PROMOTION_PATH),
    "transaction_columns": tx_columns,
    "promotion_feature_columns": promo_columns,
    "match_coverage": coverage.to_dict(orient="records"),
    "treatment_status": "NOT_ESTIMATED",
    "reason": "Only promotion code counts are available; confirmed exposure labels are not available in this processed file.",
    "limitations": [
        "Promotion code counts do not establish actual exposure status.",
        "Unmatched transactions are unknown, not untreated.",
        "DoWhy estimates and refutation tests were intentionally not run.",
        "Causal estimation requires verified treatment coding and adequate variation.",
    ],
}
with open(OUTPUT_DIR / "causal_audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, default=str)
print(json.dumps(audit, indent=2, default=str))
con.close()


{
  "project_dir": "D:\\Econocausal Double machine learning project\\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\\refutation test",
  "transaction_path": "D:\\Econocausal Double machine learning project\\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\\refutation test\\transaction_weekly_final.csv",
  "promotion_features_path": "D:\\Econocausal Double machine learning project\\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\\refutation test\\promotion_features.csv",
  "transaction_columns": [
    "PRODUCT_ID",
    "STORE_ID",
    "WEEK_NO",
    "TOTAL_QUANTITY",
    "TOTAL_SALES",
    "TOTAL_RETAIL_DISC",
    "TOTAL_COUPON_DISC",
    "TOTAL_COUPON_MATCH_DISC",
    "TRANSACTION_ROWS",
    "UNIQUE_BASKETS",
    "ZERO_QUANTITY_ROWS",
    "ZERO_SALES_ROWS",
    "EXTREME_QUANTITY_ROWS",
    "PRICE_PROXY",
    "ZERO_OR_NEGATIVE_QUANTITY_FLAG",
    "ZERO_SALES_FLAG",
    "MISSING_PRICE_PROXY_FLAG",
    "EXTREME_QUANTITY_FLAG",
    "NONPOSITIVE_PRICE_PROXY_FLAG"
  ]

In [10]:

from pathlib import Path
import pandas as pd
import json

# ============================================================
# EconoCausal: Engineered Data & Treatment Readiness Audit
# Uses existing CSV files only; does not modify source files.
# ============================================================

PROJECT_DIR = Path.cwd()

TRANSACTION_PATH = PROJECT_DIR / "transaction_weekly_final.csv"
PROMOTION_PATH = PROJECT_DIR / "promotion_features.csv"
OUTPUT_DIR = PROJECT_DIR / "causal_audit_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JOIN_KEYS = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# ---------- 1. Validate input files ----------
for path in [TRANSACTION_PATH, PROMOTION_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}\n"
            "Place this notebook in the folder containing both CSVs, "
            "or update the paths at the top of this cell."
        )

print("INPUT FILES FOUND")
print("Transactions:", TRANSACTION_PATH)
print("Promotion features:", PROMOTION_PATH)

# ---------- 2. Inspect schemas without loading full files ----------
tx_columns = pd.read_csv(TRANSACTION_PATH, nrows=0).columns.tolist()
promo_columns = pd.read_csv(PROMOTION_PATH, nrows=0).columns.tolist()

required_tx = set(JOIN_KEYS + ["TOTAL_QUANTITY"])
required_promo = set(JOIN_KEYS + ["display_code_count", "mailer_code_count"])

print("\nTRANSACTION COLUMNS:", tx_columns)
print("PROMOTION COLUMNS:", promo_columns)

if not required_tx.issubset(tx_columns):
    raise ValueError(f"Missing transaction columns: {required_tx - set(tx_columns)}")
if not required_promo.issubset(promo_columns):
    raise ValueError(f"Missing promotion columns: {required_promo - set(promo_columns)}")

# ---------- 3. Use DuckDB for scalable coverage checks ----------
try:
    import duckdb
except ImportError:
    raise ImportError(
        "DuckDB is required. Activate your existing project environment "
        "and install duckdb there if it is missing."
    )

con = duckdb.connect()
tx_path = str(TRANSACTION_PATH.resolve()).replace("\\", "/").replace("'", "''")
promo_path = str(PROMOTION_PATH.resolve()).replace("\\", "/").replace("'", "''")

con.execute(f"""
    CREATE VIEW tx AS
    SELECT PRODUCT_ID, STORE_ID, WEEK_NO, TOTAL_QUANTITY
    FROM read_csv_auto('{tx_path}')
""")
con.execute(f"""
    CREATE VIEW promo AS
    SELECT PRODUCT_ID, STORE_ID, WEEK_NO,
           display_code_count, mailer_code_count
    FROM read_csv_auto('{promo_path}')
""")

# Count unique keys and join coverage.
# This intentionally does not classify unmatched transactions as untreated.
coverage = con.execute("""
    WITH tx_keys AS (
        SELECT DISTINCT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM tx
        WHERE PRODUCT_ID IS NOT NULL
          AND STORE_ID IS NOT NULL
          AND WEEK_NO IS NOT NULL
    ),
    promo_keys AS (
        SELECT DISTINCT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM promo
        WHERE PRODUCT_ID IS NOT NULL
          AND STORE_ID IS NOT NULL
          AND WEEK_NO IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM tx_keys) AS transaction_unique_keys,
        (SELECT COUNT(*) FROM promo_keys) AS promotion_unique_keys,
        (SELECT COUNT(*) FROM tx_keys t
         INNER JOIN promo_keys p
         USING (PRODUCT_ID, STORE_ID, WEEK_NO)) AS matched_keys
""").df()

coverage["match_rate"] = (
    coverage["matched_keys"] / coverage["transaction_unique_keys"]
)
coverage["unmatched_assumed_untreated"] = False

coverage.to_csv(OUTPUT_DIR / "match_coverage_recheck.csv", index=False)

print("\nMATCH COVERAGE")
display(coverage)

# ---------- 4. Inspect code-count distributions ----------
distribution_rows = []

for column in ["display_code_count", "mailer_code_count"]:
    distribution = con.execute(f"""
        SELECT CAST({column} AS VARCHAR) AS code_count,
               COUNT(*) AS row_count
        FROM promo
        GROUP BY {column}
        ORDER BY {column}
    """).df()

    for _, row in distribution.iterrows():
        distribution_rows.append({
            "feature": column,
            "value": row["code_count"],
            "row_count": int(row["row_count"])
        })

distributions = pd.DataFrame(distribution_rows)
distributions.to_csv(
    OUTPUT_DIR / "promotion_count_distribution_recheck.csv",
    index=False
)

print("\nPROMOTION COUNT DISTRIBUTIONS")
display(distributions)

# ---------- 5. Check treatment readiness ----------
readiness = pd.DataFrame([
    {
        "candidate": "display",
        "available_feature": "display_code_count",
        "confirmed_exposure_definition": False,
        "causal_estimation_ready": False,
        "reason": (
            "Distinct-code counts do not identify the original display "
            "codes or confirm exposure."
        )
    },
    {
        "candidate": "mailer",
        "available_feature": "mailer_code_count",
        "confirmed_exposure_definition": False,
        "causal_estimation_ready": False,
        "reason": (
            "Distinct-code counts do not identify the original mailer "
            "codes or confirm exposure."
        )
    }
])

readiness.to_csv(OUTPUT_DIR / "treatment_readiness_recheck.csv", index=False)

summary = {
    "transaction_file": TRANSACTION_PATH.name,
    "promotion_file": PROMOTION_PATH.name,
    "transaction_unique_keys": int(coverage.iloc[0]["transaction_unique_keys"]),
    "promotion_unique_keys": int(coverage.iloc[0]["promotion_unique_keys"]),
    "matched_keys": int(coverage.iloc[0]["matched_keys"]),
    "match_rate": float(coverage.iloc[0]["match_rate"]),
    "unmatched_transactions_assumed_untreated": False,
    "treatment_status": "NOT_ESTIMATED",
    "reason": (
        "Existing engineered features contain code counts, not verified "
        "promotion exposure labels."
    ),
    "outputs_directory": str(OUTPUT_DIR.resolve())
}

with open(OUTPUT_DIR / "engineered_data_audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

con.close()

print("\nAUDIT COMPLETE")
print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Causal estimation remains blocked until treatment coding is verified.")


INPUT FILES FOUND
Transactions: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\transaction_weekly_final.csv
Promotion features: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\promotion_features.csv

TRANSACTION COLUMNS: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'TOTAL_QUANTITY', 'TOTAL_SALES', 'TOTAL_RETAIL_DISC', 'TOTAL_COUPON_DISC', 'TOTAL_COUPON_MATCH_DISC', 'TRANSACTION_ROWS', 'UNIQUE_BASKETS', 'ZERO_QUANTITY_ROWS', 'ZERO_SALES_ROWS', 'EXTREME_QUANTITY_ROWS', 'PRICE_PROXY', 'ZERO_OR_NEGATIVE_QUANTITY_FLAG', 'ZERO_SALES_FLAG', 'MISSING_PRICE_PROXY_FLAG', 'EXTREME_QUANTITY_FLAG', 'NONPOSITIVE_PRICE_PROXY_FLAG']
PROMOTION COLUMNS: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display_code_count', 'mailer_code_count']

MATCH COVERAGE


,transaction_unique_keys,promotion_unique_keys,matched_keys,match_rate,unmatched_assumed_untreated
0,2370784,36771279,483320,0.203865,False



PROMOTION COUNT DISTRIBUTIONS


,feature,value,row_count
0,display_code_count,1,36756071
1,display_code_count,2,15208
2,mailer_code_count,1,36771174
3,mailer_code_count,2,105



AUDIT COMPLETE
Saved outputs to: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\causal_audit_outputs
Causal estimation remains blocked until treatment coding is verified.


In [11]:

from pathlib import Path
import duckdb
import pandas as pd

# ==========================================================
# EconoCausal: Investigate the 20.39% key match rate
# Uses existing engineered CSVs; does not modify source data.
# ==========================================================

BASE = Path.cwd()
TX_PATH = BASE / "transaction_weekly_final.csv"
PROMO_PATH = BASE / "promotion_features.csv"
OUT = BASE / "causal_audit_outputs"
OUT.mkdir(parents=True, exist_ok=True)

for path in [TX_PATH, PROMO_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            "Update TX_PATH or PROMO_PATH to the correct file location."
        )

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

con = duckdb.connect()

con.execute(f"""
    CREATE VIEW tx AS
    SELECT
        CAST(PRODUCT_ID AS VARCHAR) AS PRODUCT_ID,
        CAST(STORE_ID AS VARCHAR) AS STORE_ID,
        CAST(WEEK_NO AS VARCHAR) AS WEEK_NO
    FROM read_csv_auto('{sql_path(TX_PATH)}')
""")

con.execute(f"""
    CREATE VIEW promo AS
    SELECT
        CAST(PRODUCT_ID AS VARCHAR) AS PRODUCT_ID,
        CAST(STORE_ID AS VARCHAR) AS STORE_ID,
        CAST(WEEK_NO AS VARCHAR) AS WEEK_NO
    FROM read_csv_auto('{sql_path(PROMO_PATH)}')
""")

# Unique, non-null keys only
con.execute("""
    CREATE TEMP VIEW tx_keys AS
    SELECT DISTINCT PRODUCT_ID, STORE_ID, WEEK_NO
    FROM tx
    WHERE PRODUCT_ID IS NOT NULL
      AND STORE_ID IS NOT NULL
      AND WEEK_NO IS NOT NULL
""")

con.execute("""
    CREATE TEMP VIEW promo_keys AS
    SELECT DISTINCT PRODUCT_ID, STORE_ID, WEEK_NO
    FROM promo
    WHERE PRODUCT_ID IS NOT NULL
      AND STORE_ID IS NOT NULL
      AND WEEK_NO IS NOT NULL
""")

# 1. Check overlap for each key column separately
single_key_results = []

for key in ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]:
    query = f"""
        WITH
        t AS (SELECT DISTINCT {key} AS k FROM tx_keys),
        p AS (SELECT DISTINCT {key} AS k FROM promo_keys)
        SELECT
            (SELECT COUNT(*) FROM t) AS transaction_values,
            (SELECT COUNT(*) FROM p) AS promotion_values,
            (SELECT COUNT(*) FROM t INNER JOIN p USING (k))
                AS overlapping_values
    """
    result = con.execute(query).fetchone()
    single_key_results.append({
        "key": key,
        "transaction_values": result[0],
        "promotion_values": result[1],
        "overlapping_values": result[2],
        "transaction_value_overlap_rate":
            result[2] / result[0] if result[0] else None
    })

single_key = pd.DataFrame(single_key_results)
single_key.to_csv(OUT / "key_value_overlap.csv", index=False)

print("1. INDIVIDUAL KEY VALUE OVERLAP")
display(single_key)

# 2. Compare transaction and promotion week coverage
weekly = con.execute("""
    WITH
    tx_week AS (
        SELECT WEEK_NO, COUNT(*) AS transaction_keys
        FROM tx_keys
        GROUP BY WEEK_NO
    ),
    promo_week AS (
        SELECT WEEK_NO, COUNT(*) AS promotion_keys
        FROM promo_keys
        GROUP BY WEEK_NO
    ),
    matched_week AS (
        SELECT t.WEEK_NO, COUNT(*) AS matched_keys
        FROM tx_keys t
        INNER JOIN promo_keys p
        USING (PRODUCT_ID, STORE_ID, WEEK_NO)
        GROUP BY t.WEEK_NO
    )
    SELECT
        t.WEEK_NO,
        t.transaction_keys,
        COALESCE(p.promotion_keys, 0) AS promotion_keys,
        COALESCE(m.matched_keys, 0) AS matched_keys,
        COALESCE(m.matched_keys, 0) * 1.0
            / NULLIF(t.transaction_keys, 0) AS match_rate
    FROM tx_week t
    LEFT JOIN promo_week p USING (WEEK_NO)
    LEFT JOIN matched_week m USING (WEEK_NO)
    ORDER BY TRY_CAST(t.WEEK_NO AS INTEGER), t.WEEK_NO
""").df()

weekly.to_csv(OUT / "match_coverage_by_week.csv", index=False)

print("\n2. WEEK-LEVEL MATCH COVERAGE")
display(weekly.head(20))

# 3. Overall summary, including null key counts in original files
summary = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM tx_keys) AS transaction_unique_keys,
        (SELECT COUNT(*) FROM promo_keys) AS promotion_unique_keys,
        (SELECT COUNT(*) FROM tx_keys t
         INNER JOIN promo_keys p
         USING (PRODUCT_ID, STORE_ID, WEEK_NO)) AS matched_keys,
        (SELECT COUNT(*) FROM tx
         WHERE PRODUCT_ID IS NULL OR STORE_ID IS NULL OR WEEK_NO IS NULL)
            AS transaction_rows_with_null_keys,
        (SELECT COUNT(*) FROM promo
         WHERE PRODUCT_ID IS NULL OR STORE_ID IS NULL OR WEEK_NO IS NULL)
            AS promotion_rows_with_null_keys
""").df()

summary["match_rate"] = (
    summary["matched_keys"] / summary["transaction_unique_keys"]
)

summary.to_csv(OUT / "key_overlap_summary.csv", index=False)

print("\n3. OVERALL SUMMARY")
display(summary)

con.close()

print("\nDiagnostic complete. Files saved in:")
print(OUT.resolve())


1. INDIVIDUAL KEY VALUE OVERLAP


,key,transaction_values,promotion_values,overlapping_values,transaction_value_overlap_rate
0,PRODUCT_ID,92339,68377,68377,0.740500
1,STORE_ID,582,115,115,0.197595
2,WEEK_NO,102,93,93,0.911765



2. WEEK-LEVEL MATCH COVERAGE


,WEEK_NO,transaction_keys,promotion_keys,matched_keys,match_rate
0,1,1855,0,0,0.000000
1,2,3597,0,0,0.000000
2,3,4630,0,0,0.000000
3,4,5175,0,0,0.000000
4,5,6866,0,0,0.000000
5,6,8514,0,0,0.000000
6,7,8620,0,0,0.000000
7,8,9908,0,0,0.000000
8,9,9967,398116,2326,0.233370
9,10,12723,346537,2576,0.202468



3. OVERALL SUMMARY


,transaction_unique_keys,promotion_unique_keys,matched_keys,transaction_rows_with_null_keys,promotion_rows_with_null_keys,match_rate
0,2370784,36771279,483320,0,0,0.203865



Diagnostic complete. Files saved in:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\causal_audit_outputs


In [12]:

from pathlib import Path
import duckdb

# Store-level mismatch diagnostic only
BASE = Path.cwd()
TX_PATH = BASE / "transaction_weekly_final.csv"
PROMO_PATH = BASE / "promotion_features.csv"
OUT = BASE / "causal_audit_outputs"
OUT.mkdir(parents=True, exist_ok=True)

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

for path in (TX_PATH, PROMO_PATH):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

con = duckdb.connect()

con.execute(f"""
    CREATE VIEW tx AS
    SELECT
        CAST(STORE_ID AS VARCHAR) AS STORE_ID,
        PRODUCT_ID,
        WEEK_NO
    FROM read_csv_auto('{sql_path(TX_PATH)}')
""")

con.execute(f"""
    CREATE VIEW promo AS
    SELECT
        CAST(STORE_ID AS VARCHAR) AS STORE_ID,
        PRODUCT_ID,
        WEEK_NO
    FROM read_csv_auto('{sql_path(PROMO_PATH)}')
""")

# Count unique transaction keys per store
# and flag whether each store appears in promotion data.
store_audit = con.execute("""
    WITH tx_store AS (
        SELECT
            STORE_ID,
            COUNT(DISTINCT (PRODUCT_ID, WEEK_NO)) AS transaction_keys
        FROM tx
        WHERE STORE_ID IS NOT NULL
          AND PRODUCT_ID IS NOT NULL
          AND WEEK_NO IS NOT NULL
        GROUP BY STORE_ID
    ),
    promo_store AS (
        SELECT DISTINCT STORE_ID
        FROM promo
        WHERE STORE_ID IS NOT NULL
    )
    SELECT
        t.STORE_ID,
        t.transaction_keys,
        CASE
            WHEN p.STORE_ID IS NULL THEN 'ABSENT_FROM_PROMOTION_FILE'
            ELSE 'PRESENT_IN_PROMOTION_FILE'
        END AS store_status
    FROM tx_store t
    LEFT JOIN promo_store p USING (STORE_ID)
    ORDER BY
        CASE WHEN p.STORE_ID IS NULL THEN 0 ELSE 1 END,
        t.transaction_keys DESC
""").df()

store_audit.to_csv(OUT / "store_id_mismatch_diagnostic.csv", index=False)

print("STORE STATUS SUMMARY")
print(store_audit.groupby("store_status")["transaction_keys"].agg(
    store_count="count",
    transaction_keys="sum"
).to_string())

print("\nSTORES ABSENT FROM PROMOTION FILE (first 30)")
print(
    store_audit[
        store_audit["store_status"] == "ABSENT_FROM_PROMOTION_FILE"
    ].head(30).to_string(index=False)
)

con.close()

print("\nSaved:", OUT / "store_id_mismatch_diagnostic.csv")


STORE STATUS SUMMARY
                            store_count  transaction_keys
store_status                                             
ABSENT_FROM_PROMOTION_FILE          467             36509
PRESENT_IN_PROMOTION_FILE           115           2334275

STORES ABSENT FROM PROMOTION FILE (first 30)
STORE_ID  transaction_keys               store_status
     673              2695 ABSENT_FROM_PROMOTION_FILE
    3235              1564 ABSENT_FROM_PROMOTION_FILE
    3001              1176 ABSENT_FROM_PROMOTION_FILE
    3182              1135 ABSENT_FROM_PROMOTION_FILE
     569              1056 ABSENT_FROM_PROMOTION_FILE
   31863               913 ABSENT_FROM_PROMOTION_FILE
     514               874 ABSENT_FROM_PROMOTION_FILE
     773               855 ABSENT_FROM_PROMOTION_FILE
      27               825 ABSENT_FROM_PROMOTION_FILE
    3297               750 ABSENT_FROM_PROMOTION_FILE
     627               642 ABSENT_FROM_PROMOTION_FILE
    3274               596 ABSENT_FROM_PROMOTION_FILE

In [16]:

from pathlib import Path
import duckdb

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------
BASE = Path.cwd()
TX_PATH = BASE / "transaction_weekly_final.csv"
PROMO_PATH = BASE / "promotion_features.csv"
OUT = BASE / "causal_audit_outputs"
OUT.mkdir(parents=True, exist_ok=True)

for path in (TX_PATH, PROMO_PATH):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

# Use disk-backed temporary storage to reduce RAM pressure.
# Temporary files will be stored alongside the project.
con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")
con.execute(f"SET temp_directory = '{sql_path(OUT)}/duckdb_temp'")

tx_file = sql_path(TX_PATH)
promo_file = sql_path(PROMO_PATH)

# --------------------------------------------------
# 2. Read only keys and normalize their data types
# --------------------------------------------------
con.execute(f"""
    CREATE TEMP TABLE tx AS
    SELECT
        TRIM(CAST(PRODUCT_ID AS VARCHAR)) AS PRODUCT_ID,
        TRIM(CAST(STORE_ID AS VARCHAR)) AS STORE_ID,
        TRY_CAST(WEEK_NO AS INTEGER) AS WEEK_NO
    FROM read_csv(
        '{tx_file}',
        header = true,
        all_varchar = true
    )
    WHERE PRODUCT_ID IS NOT NULL
      AND STORE_ID IS NOT NULL
      AND WEEK_NO IS NOT NULL
""")

print("Transaction keys loaded.")

con.execute(f"""
    CREATE TEMP TABLE promo AS
    SELECT
        TRIM(CAST(PRODUCT_ID AS VARCHAR)) AS PRODUCT_ID,
        TRIM(CAST(STORE_ID AS VARCHAR)) AS STORE_ID,
        TRY_CAST(WEEK_NO AS INTEGER) AS WEEK_NO
    FROM read_csv(
        '{promo_file}',
        header = true,
        all_varchar = true
    )
    WHERE PRODUCT_ID IS NOT NULL
      AND STORE_ID IS NOT NULL
      AND WEEK_NO IS NOT NULL
""")

print("Promotion keys loaded.")

# --------------------------------------------------
# 3. Deduplicate keys and find exact matches
# --------------------------------------------------
con.execute("""
    CREATE TEMP TABLE tx_keys AS
    SELECT DISTINCT PRODUCT_ID, STORE_ID, WEEK_NO
    FROM tx
""")

con.execute("""
    CREATE TEMP TABLE promo_keys AS
    SELECT DISTINCT PRODUCT_ID, STORE_ID, WEEK_NO
    FROM promo
""")

print("Unique keys prepared.")

# --------------------------------------------------
# 4. Match transaction keys to promotion keys
# --------------------------------------------------
con.execute("""
    CREATE TEMP TABLE matched_keys AS
    SELECT t.PRODUCT_ID, t.STORE_ID, t.WEEK_NO
    FROM tx_keys t
    INNER JOIN promo_keys p
    USING (PRODUCT_ID, STORE_ID, WEEK_NO)
""")

print("Exact key matching completed.")

# --------------------------------------------------
# 5. Store/week coverage summary
# --------------------------------------------------

weekly_output = sql_path(OUT / "shared_store_week_coverage.csv")

con.execute(f"""
    COPY (
        WITH
        tx_week AS (
            SELECT STORE_ID, WEEK_NO, COUNT(*) AS transaction_keys
            FROM tx_keys
            GROUP BY STORE_ID, WEEK_NO
        ),
        promo_week AS (
            SELECT STORE_ID, WEEK_NO, COUNT(*) AS promotion_keys
            FROM promo_keys
            GROUP BY STORE_ID, WEEK_NO
        ),
        matched_week AS (
            SELECT STORE_ID, WEEK_NO, COUNT(*) AS matched_keys
            FROM matched_keys
            GROUP BY STORE_ID, WEEK_NO
        )
        SELECT
            t.STORE_ID,
            t.WEEK_NO,
            t.transaction_keys,
            COALESCE(p.promotion_keys, 0) AS promotion_keys,
            COALESCE(m.matched_keys, 0) AS matched_keys,
            COALESCE(m.matched_keys, 0) * 1.0
                / NULLIF(t.transaction_keys, 0) AS match_rate
        FROM tx_week t
        LEFT JOIN promo_week p USING (STORE_ID, WEEK_NO)
        LEFT JOIN matched_week m USING (STORE_ID, WEEK_NO)
    )
    TO '{weekly_output}'
    (HEADER, DELIMITER ',')
""")

print("Store/week coverage saved to:", weekly_output)



# --------------------------------------------------
# 6. Store-level summary
# --------------------------------------------------
con.execute("""
    CREATE TEMP TABLE store_summary AS
    SELECT
        t.STORE_ID,
        COUNT(*) AS transaction_keys,
        COUNT(m.PRODUCT_ID) AS matched_keys,
        COUNT(m.PRODUCT_ID) * 1.0 / NULLIF(COUNT(*), 0)
            AS match_rate
    FROM tx_keys t
    LEFT JOIN matched_keys m
    USING (PRODUCT_ID, STORE_ID, WEEK_NO)
    GROUP BY t.STORE_ID
""")

con.execute(f"""
    COPY (
        SELECT *
        FROM store_summary
        ORDER BY match_rate ASC
    )
    TO '{sql_path(OUT / "shared_store_coverage.csv")}'
    (HEADER, DELIMITER ',')
""")

# --------------------------------------------------
# 7. Overall matching summary
# --------------------------------------------------
overall = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM tx_keys) AS transaction_keys,
        (SELECT COUNT(*) FROM promo_keys) AS promotion_keys,
        (SELECT COUNT(*) FROM matched_keys) AS matched_keys
""").fetchone()

tx_count, promo_count, matched_count = overall
overall_rate = matched_count / tx_count if tx_count else 0

print("\nOVERALL KEY COVERAGE")
print(f"Transaction keys: {tx_count:,}")
print(f"Promotion keys:   {promo_count:,}")
print(f"Matched keys:     {matched_count:,}")
print(f"Match rate:       {overall_rate:.2%}")

print("\nLOWEST-MATCHING STORES")
display(con.execute("""
    SELECT *
    FROM store_summary
    ORDER BY match_rate ASC
    LIMIT 15
""").df())

print("\nLOWEST-MATCHING STORE/WEEK COMBINATIONS")
display(con.execute(f"""
    SELECT *
    FROM read_csv_auto(
        '{sql_path(OUT / "shared_store_week_coverage.csv")}'
    )
    ORDER BY match_rate ASC, transaction_keys DESC
    LIMIT 20
""").df())

print("\nSaved results to:", OUT.resolve())

con.close()



Transaction keys loaded.
Promotion keys loaded.
Unique keys prepared.
Exact key matching completed.
Store/week coverage saved to: D:/Econocausal Double machine learning project/EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning/refutation test/causal_audit_outputs/shared_store_week_coverage.csv

OVERALL KEY COVERAGE
Transaction keys: 2,370,784
Promotion keys:   36,771,279
Matched keys:     483,320
Match rate:       20.39%

LOWEST-MATCHING STORES


,STORE_ID,transaction_keys,matched_keys,match_rate
0,2817,133,0,0.0
1,2903,148,0,0.0
2,2959,14,0,0.0
3,2888,591,0,0.0
4,2971,357,0,0.0
5,2863,392,0,0.0
6,2852,249,0,0.0
7,2946,351,0,0.0
8,2786,52,0,0.0
9,2918,220,0,0.0



LOWEST-MATCHING STORE/WEEK COMBINATIONS


,STORE_ID,WEEK_NO,transaction_keys,promotion_keys,matched_keys,match_rate
0,367,7,517,0,0,0.0
1,367,4,493,0,0,0.0
2,367,102,447,0,0,0.0
3,406,102,418,0,0,0.0
4,410,102,415,0,0,0.0
5,34280,102,403,0,0,0.0
6,427,102,400,0,0,0.0
7,321,102,379,0,0,0.0
8,356,102,374,0,0,0.0
9,381,102,370,0,0,0.0



Saved results to: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\causal_audit_outputs


In [17]:
from pathlib import Path
import pandas as pd

OUT = Path.cwd() / "causal_audit_outputs"
weekly_path = OUT / "shared_store_week_coverage.csv"

weekly = pd.read_csv(
    weekly_path,
    usecols=[
        "transaction_keys",
        "promotion_keys",
        "matched_keys",
        "match_rate"
    ]
)

print("STORE/WEEK MATCHING SUMMARY")
print("-" * 40)
print(f"Total store/week combinations: {len(weekly):,}")
print(
    "No promotion keys:",
    f"{(weekly['promotion_keys'] == 0).sum():,}"
)
print(
    "Promotion keys exist, but no exact matches:",
    f"{((weekly['promotion_keys'] > 0) & (weekly['matched_keys'] == 0)).sum():,}"
)
print(
    "Partial matching:",
    f"{((weekly['matched_keys'] > 0) & (weekly['match_rate'] < 1)).sum():,}"
)
print(
    "Complete matching:",
    f"{(weekly['match_rate'] == 1).sum():,}"
)

print("\nMATCH RATE DISTRIBUTION")
display(
    weekly["match_rate"]
    .describe()
    .to_frame("match_rate")
)

STORE/WEEK MATCHING SUMMARY
----------------------------------------
Total store/week combinations: 13,939
No promotion keys: 3,901
Promotion keys exist, but no exact matches: 13
Partial matching: 10,023
Complete matching: 2

MATCH RATE DISTRIBUTION


,match_rate
count,13939.000000
mean,0.157885
std,0.112233
min,0.000000
25%,0.000000
50%,0.185345
75%,0.236686
max,1.000000


In [18]:
from pathlib import Path
import pandas as pd

OUT = Path.cwd() / "causal_audit_outputs"

weekly = pd.read_csv(
    OUT / "shared_store_week_coverage.csv"
)

# Aggregate the existing store/week results
total_transaction_keys = weekly["transaction_keys"].sum()
total_matched_keys = weekly["matched_keys"].sum()

summary = pd.DataFrame({
    "Metric": [
        "Transaction keys",
        "Matched keys",
        "Unmatched keys",
        "Match rate",
        "Unmatched rate"
    ],
    "Value": [
        total_transaction_keys,
        total_matched_keys,
        total_transaction_keys - total_matched_keys,
        total_matched_keys / total_transaction_keys,
        1 - total_matched_keys / total_transaction_keys
    ]
})

display(summary)

,Metric,Value
0,Transaction keys,2.370784e+06
1,Matched keys,4.833200e+05
2,Unmatched keys,1.887464e+06
3,Match rate,2.038651e-01
4,Unmatched rate,7.961349e-01


In [19]:
from pathlib import Path
import duckdb

BASE = Path.cwd()

TX_PATH = BASE / "transaction_weekly_final.csv"
PROMO_PATH = BASE / "promotion_features.csv"

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

con = duckdb.connect()

for name, path in [
    ("TRANSACTIONS", TX_PATH),
    ("PROMOTIONS", PROMO_PATH)
]:
    print(f"\n{name}: KEY COLUMN TYPES")

    result = con.execute(f"""
        DESCRIBE
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM read_csv_auto('{sql_path(path)}')
    """).df()

    display(result)

    print(f"\n{name}: SAMPLE KEY VALUES")

    sample = con.execute(f"""
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO
        FROM read_csv_auto('{sql_path(path)}')
        LIMIT 5
    """).df()

    display(sample)

con.close()


TRANSACTIONS: KEY COLUMN TYPES


,column_name,column_type,null,key,default,extra
0,PRODUCT_ID,BIGINT,YES,None,None,None
1,STORE_ID,BIGINT,YES,None,None,None
2,WEEK_NO,BIGINT,YES,None,None,None



TRANSACTIONS: SAMPLE KEY VALUES


,PRODUCT_ID,STORE_ID,WEEK_NO
0,25671,3191,59
1,25671,3266,36
2,25671,3313,23
3,26081,3235,36
4,26093,33904,66



PROMOTIONS: KEY COLUMN TYPES


,column_name,column_type,null,key,default,extra
0,PRODUCT_ID,BIGINT,YES,None,None,None
1,STORE_ID,BIGINT,YES,None,None,None
2,WEEK_NO,BIGINT,YES,None,None,None



PROMOTIONS: SAMPLE KEY VALUES


,PRODUCT_ID,STORE_ID,WEEK_NO
0,26190,286,70
1,26190,288,70
2,26190,289,70
3,26190,292,70
4,26190,293,70


In [21]:

from pathlib import Path
import duckdb

BASE = Path.cwd()
TX_PATH = BASE / "transaction_weekly_final.csv"
PROMO_PATH = BASE / "promotion_features.csv"

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

con = duckdb.connect()
con.execute("SET threads = 2")

tx_file = sql_path(TX_PATH)
promo_file = sql_path(PROMO_PATH)

# Check product overlap
product_result = con.execute(f"""
    WITH
    tx_products AS (
        SELECT DISTINCT PRODUCT_ID
        FROM read_csv_auto('{tx_file}')
        WHERE PRODUCT_ID IS NOT NULL
    ),
    promo_products AS (
        SELECT DISTINCT PRODUCT_ID
        FROM read_csv_auto('{promo_file}')
        WHERE PRODUCT_ID IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM tx_products) AS transaction_products,
        (SELECT COUNT(*) FROM promo_products) AS promotion_products,
        (
            SELECT COUNT(*)
            FROM tx_products t
            INNER JOIN promo_products p USING (PRODUCT_ID)
        ) AS shared_products
""").df()

print("PRODUCT OVERLAP")
display(product_result)

# Check store overlap
store_result = con.execute(f"""
    WITH
    tx_stores AS (
        SELECT DISTINCT STORE_ID
        FROM read_csv_auto('{tx_file}')
        WHERE STORE_ID IS NOT NULL
    ),
    promo_stores AS (
        SELECT DISTINCT STORE_ID
        FROM read_csv_auto('{promo_file}')
        WHERE STORE_ID IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM tx_stores) AS transaction_stores,
        (SELECT COUNT(*) FROM promo_stores) AS promotion_stores,
        (
            SELECT COUNT(*)
            FROM tx_stores t
            INNER JOIN promo_stores p USING (STORE_ID)
        ) AS shared_stores
""").df()

print("\nSTORE OVERLAP")
display(store_result)

con.close()



PRODUCT OVERLAP


,transaction_products,promotion_products,shared_products
0,92339,68377,68377



STORE OVERLAP


,transaction_stores,promotion_stores,shared_stores
0,582,115,115


In [22]:
from pathlib import Path
import duckdb

BASE = Path.cwd()
TX_PATH = BASE / "transaction_weekly_final.csv"
PROMO_PATH = BASE / "promotion_features.csv"

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

con = duckdb.connect()
con.execute("SET threads = 2")

for name, path in [
    ("TRANSACTIONS", TX_PATH),
    ("PROMOTIONS", PROMO_PATH)
]:
    result = con.execute(f"""
        SELECT
            MIN(WEEK_NO) AS first_week,
            MAX(WEEK_NO) AS last_week,
            COUNT(DISTINCT WEEK_NO) AS number_of_weeks
        FROM read_csv_auto('{sql_path(path)}')
    """).df()

    print(f"\n{name}: WEEK COVERAGE")
    display(result)

con.close()


TRANSACTIONS: WEEK COVERAGE


,first_week,last_week,number_of_weeks
0,1,102,102



PROMOTIONS: WEEK COVERAGE


,first_week,last_week,number_of_weeks
0,9,101,93


In [23]:

from pathlib import Path
import duckdb
import json

BASE = Path.cwd()
PROMO_PATH = BASE / "promotion_features.csv"
TX_PATH = BASE / "transaction_weekly_final.csv"
OUT = BASE / "causal_audit_outputs"
OUT.mkdir(parents=True, exist_ok=True)

def sql_path(path):
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

for path in [PROMO_PATH, TX_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

con = duckdb.connect()
con.execute("SET threads = 2")

promo_file = sql_path(PROMO_PATH)
tx_file = sql_path(TX_PATH)

# 1. Check available promotion feature columns
print("1. AVAILABLE PROMOTION COLUMNS")
display(con.execute(f"""
    DESCRIBE SELECT *
    FROM read_csv_auto('{promo_file}')
""").df())

# 2. Inspect the distributions of the existing code counts
print("\n2. DISPLAY CODE COUNT DISTRIBUTION")
display(con.execute(f"""
    SELECT
        display_code_count,
        COUNT(*) AS row_count
    FROM read_csv_auto('{promo_file}')
    GROUP BY display_code_count
    ORDER BY display_code_count
""").df())

print("\n3. MAILER CODE COUNT DISTRIBUTION")
display(con.execute(f"""
    SELECT
        mailer_code_count,
        COUNT(*) AS row_count
    FROM read_csv_auto('{promo_file}')
    GROUP BY mailer_code_count
    ORDER BY mailer_code_count
""").df())

# 3. Check missing values in the available count fields
print("\n4. MISSING PROMOTION FEATURE VALUES")
display(con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE display_code_count IS NULL
        ) AS missing_display_counts,
        COUNT(*) FILTER (
            WHERE mailer_code_count IS NULL
        ) AS missing_mailer_counts
    FROM read_csv_auto('{promo_file}')
""").df())

# 4. Record what we can and cannot conclude
report = {
    "promotion_file": PROMO_PATH.name,
    "transaction_file": TX_PATH.name,
    "treatment_readiness": "NOT_VERIFIED",
    "reason": (
        "The processed file contains distinct-code counts. "
        "The original display and mailer code meanings must be "
        "verified before defining treatment exposure."
    ),
    "causal_effect_isolated": False
}

report_path = OUT / "treatment_readiness_check.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n5. CURRENT TREATMENT READINESS")
print(json.dumps(report, indent=2))
print("\nSaved report to:", report_path)

con.close()



1. AVAILABLE PROMOTION COLUMNS


,column_name,column_type,null,key,default,extra
0,PRODUCT_ID,BIGINT,YES,None,None,None
1,STORE_ID,BIGINT,YES,None,None,None
2,WEEK_NO,BIGINT,YES,None,None,None
3,display_code_count,BIGINT,YES,None,None,None
4,mailer_code_count,BIGINT,YES,None,None,None



2. DISPLAY CODE COUNT DISTRIBUTION


,display_code_count,row_count
0,1,36756071
1,2,15208



3. MAILER CODE COUNT DISTRIBUTION


,mailer_code_count,row_count
0,1,36771174
1,2,105



4. MISSING PROMOTION FEATURE VALUES


,total_rows,missing_display_counts,missing_mailer_counts
0,36771279,0,0



5. CURRENT TREATMENT READINESS
{
  "promotion_file": "promotion_features.csv",
  "transaction_file": "transaction_weekly_final.csv",
  "treatment_readiness": "NOT_VERIFIED",
  "reason": "The processed file contains distinct-code counts. The original display and mailer code meanings must be verified before defining treatment exposure.",
  "causal_effect_isolated": false
}

Saved report to: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\causal_audit_outputs\treatment_readiness_check.json


In [24]:

from pathlib import Path
import json

BASE = Path.cwd()

# Search the project folder and its immediate subfolders.
# Do not read or load the large CSV files.
candidates = [
    BASE,
    *[p for p in BASE.iterdir() if p.is_dir()]
]

target_names = {
    "causal_data.csv",
    "promotions.csv",
    "promotion.csv",
    "Feature_Engineering(4).ipynb",
}

found = []

for folder in candidates:
    for path in folder.iterdir():
        if path.is_file() and path.name in target_names:
            found.append({
                "file": path.name,
                "path": str(path.resolve()),
                "size_mb": round(path.stat().st_size / (1024**2), 2)
            })

print("ORIGINAL SOURCE FILE CHECK")
print("-" * 40)

if found:
    for item in found:
        print(
            f"File: {item['file']}\n"
            f"Path: {item['path']}\n"
            f"Size: {item['size_mb']} MB\n"
        )
else:
    print("No matching source files found in the checked folders.")

report = {
    "purpose": "Locate original promotion source for treatment validation",
    "files_found": found,
    "treatment_verified": False,
    "causal_effect_isolated": False
}

out = BASE / "causal_audit_outputs"
out.mkdir(parents=True, exist_ok=True)

report_path = out / "causal_source_availability.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Report saved to:", report_path)
print("\nTreatment definition remains unverified until source semantics are established.")



ORIGINAL SOURCE FILE CHECK
----------------------------------------
No matching source files found in the checked folders.
Report saved to: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\causal_audit_outputs\causal_source_availability.json

Treatment definition remains unverified until source semantics are established.


In [26]:

from pathlib import Path
import importlib.util

print("KAGGLE API CHECK")
print("-" * 35)

# Check whether the Kaggle package is installed
installed = importlib.util.find_spec("kaggle") is not None
print("Kaggle package installed:", installed)

# Check whether Kaggle credentials exist
credential_paths = [
    Path.home() / ".kaggle" / "kaggle.json",
    Path.home() / ".config" / "kaggle" / "access_token",
]

for path in credential_paths:
    print(f"Credentials found at {path}: {path.exists()}")

if not installed:
    print("\nNext step: install the Kaggle package in your active environment.")
elif not any(p.exists() for p in credential_paths):
    print("\nNext step: configure Kaggle API authentication.")
else:
    print("\nKaggle package and a possible credential file are present.")


KAGGLE API CHECK
-----------------------------------
Kaggle package installed: False
Credentials found at C:\Users\B Ramesh\.kaggle\kaggle.json: False
Credentials found at C:\Users\B Ramesh\.config\kaggle\access_token: False

Next step: install the Kaggle package in your active environment.


In [27]:
import subprocess
import sys

print("Python environment:", sys.executable)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "kaggle"
])

print("\nKaggle package installation completed.")

Python environment: d:\EconoCausal_Env\env\python.exe

Kaggle package installation completed.


In [28]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "kaggle", "--version"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout or result.stderr)

Return code: 0
Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
Kaggle CLI 2.2.5



In [29]:
import os
import getpass
import subprocess
import sys

token = getpass.getpass("Paste your NEW Kaggle API token: ")

if not token.strip():
    raise ValueError("No token provided.")

os.environ["KAGGLE_API_TOKEN"] = token.strip()
del token

print("Kaggle token configured for this notebook session.")

Kaggle token configured for this notebook session.


In [30]:
import subprocess
import sys
import os

dataset = "frtgnn/dunnhumby-the-complete-journey"

result = subprocess.run(
    [
        sys.executable, "-m", "kaggle",
        "datasets", "files", dataset
    ],
    capture_output=True,
    text=True,
    env=os.environ.copy()
)

print("Return code:", result.returncode)
print(result.stdout if result.returncode == 0 else result.stderr)

Return code: 0
name                       size  creationDate                
--------------------  ---------  --------------------------  
campaign_desc.csv           540  2019-11-07 06:59:03.908000  
campaign_table.csv        95874  2019-11-07 06:59:03.928000  
causal_data.csv       695858427  2019-11-07 06:59:25.548000  
coupon.csv              2822804  2019-11-07 06:59:12.723000  
coupon_redempt.csv        54108  2019-11-07 06:59:12.642000  
hh_demographic.csv        44349  2019-11-07 06:59:12.522000  
product.csv             6429896  2019-11-07 06:59:12.584000  
transaction_data.csv  141742346  2019-11-07 06:59:14.969000  



In [32]:
import subprocess
import sys
import os

dataset = "frtgnn/dunnhumby-the-complete-journey"

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "kaggle",
        "datasets",
        "metadata",
        dataset
    ],
    capture_output=True,
    text=True,
    env=os.environ.copy(),
    timeout=60
)

print("Return code:", result.returncode)

if result.returncode == 0:
    print(result.stdout)
else:
    print(result.stderr)

Return code: 0
Downloaded metadata to d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\dataset-metadata.json



In [33]:
from pathlib import Path
import json

BASE = Path.cwd()

# Find metadata files downloaded by the Kaggle CLI
metadata_files = list(BASE.rglob("dataset-metadata.json"))

if not metadata_files:
    print("No dataset-metadata.json found in the project folder.")
    print("Current working directory:", BASE)
else:
    for path in metadata_files:
        print(f"\nMetadata file: {path}")
        try:
            metadata = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(metadata, indent=2))
        except (json.JSONDecodeError, OSError) as e:
            print("Could not read metadata:", e)


Metadata file: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\dataset-metadata.json
{
  "info": {
    "datasetId": 408408,
    "datasetSlug": "dunnhumby-the-complete-journey",
    "ownerUser": "frtgnn",
    "usabilityRating": 1,
    "totalViews": 91855,
    "totalVotes": 207,
    "totalDownloads": 18111,
    "title": "Dunnhumby - The Complete Journey",
    "subtitle": "Retail focused consumer data ",
    "description": "This dataset contains household level transactions over two years from a group of 2,500 households who are frequent shoppers at a retailer. It contains all of each household\u2019s purchases, not just those from a limited number of categories. For certain households, demographic information as well as direct marketing contact history are included.\n\nDue to the number of tables and the overall complexity of The Complete Journey, it is suggested that this database be used in more advanced classroom 

In [34]:

import os
import subprocess
import sys

dataset = "frtgnn/dunnhumby-the-complete-journey"

result = subprocess.run(
    [
        sys.executable,
        "-m", "kaggle",
        "datasets", "files", dataset
    ],
    capture_output=True,
    text=True,
    env=os.environ.copy(),
    timeout=60
)

if result.returncode != 0:
    print(result.stderr)
else:
    print("FILES AVAILABLE IN THE ORIGINAL DATASET\n")
    print(result.stdout)

    print(
        "\nLook for documentation files such as README, "
        "dictionary, documentation, or PDF files."
    )



FILES AVAILABLE IN THE ORIGINAL DATASET

name                       size  creationDate                
--------------------  ---------  --------------------------  
campaign_desc.csv           540  2019-11-07 06:59:03.908000  
campaign_table.csv        95874  2019-11-07 06:59:03.928000  
causal_data.csv       695858427  2019-11-07 06:59:25.548000  
coupon.csv              2822804  2019-11-07 06:59:12.723000  
coupon_redempt.csv        54108  2019-11-07 06:59:12.642000  
hh_demographic.csv        44349  2019-11-07 06:59:12.522000  
product.csv             6429896  2019-11-07 06:59:12.584000  
transaction_data.csv  141742346  2019-11-07 06:59:14.969000  


Look for documentation files such as README, dictionary, documentation, or PDF files.


In [35]:

import os
import subprocess
import sys
from pathlib import Path

DATASET = "frtgnn/dunnhumby-the-complete-journey"
OUT = Path.cwd() / "source_metadata"
OUT.mkdir(parents=True, exist_ok=True)

files_to_get = ["campaign_desc.csv", "campaign_table.csv"]

for filename in files_to_get:
    print(f"\nRetrieving {filename}...")

    result = subprocess.run(
        [
            sys.executable,
            "-m", "kaggle",
            "datasets", "download",
            "-d", DATASET,
            "-f", filename,
            "-p", str(OUT),
            "--unzip"
        ],
        capture_output=True,
        text=True,
        env=os.environ.copy(),
        timeout=120
    )

    if result.returncode == 0:
        print(f"Downloaded: {OUT / filename}")
    else:
        print(result.stderr)

    path = OUT / filename
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="replace")[:4000])



Retrieving campaign_desc.csv...
Downloaded: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\source_metadata\campaign_desc.csv
DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
TypeB,24,659,719
TypeC,15,547,708
TypeB,25,659,691
TypeC,20,615,685
TypeB,23,646,684
TypeB,21,624,656
TypeB,22,624,656
TypeA,18,587,642
TypeB,19,603,635
TypeB,17,575,607
TypeC,14,531,596
TypeB,16,561,593
TypeA,13,504,551
TypeB,11,477,523
TypeB,12,477,509
TypeB,10,463,495
TypeB,9,435,467
TypeA,8,412,460
TypeB,7,398,432
TypeC,6,393,425
TypeC,3,356,412
TypeB,5,377,411
TypeB,4,372,404
TypeB,1,346,383
TypeB,2,351,383
TypeA,30,323,369
TypeB,29,281,334
TypeB,28,259,320
TypeC,27,237,300
TypeA,26,224,264


Retrieving campaign_table.csv...
Downloaded: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\source_metadata\campaign_table.csv
DESCRIPTION,household_key,CAMPAIGN
TypeA,17,26
TypeA,27,26
Ty

In [36]:

from pathlib import Path
import json

BASE = Path.cwd()

notebooks = list(BASE.rglob("Feature_Engineering*.ipynb"))

if not notebooks:
    print("No Feature_Engineering notebook found under:", BASE)
else:
    for notebook_path in notebooks:
        print(f"\nNOTEBOOK: {notebook_path}")

        with notebook_path.open("r", encoding="utf-8") as f:
            notebook = json.load(f)

        for i, cell in enumerate(notebook.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            if any(term in source.lower() for term in [
                "causal_data.csv",
                'promotions["display"]',
                "promotions['display']",
                'promotions["mailer"]',
                "promotions['mailer']",
                "display_code_count",
                "mailer_code_count"
            ]):
                print(f"\n--- CELL {i} ---")
                print(source[:5000])



No Feature_Engineering notebook found under: d:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test


In [37]:

from pathlib import Path

PROJECT = Path(
    r"D:\Econocausal Double machine learning project"
)

print("Searching for Feature_Engineering notebooks...")

matches = list(PROJECT.rglob("Feature_Engineering*.ipynb"))

if matches:
    for path in matches:
        print(f"\nFound: {path}")
        print(f"Size: {path.stat().st_size / 1024:.1f} KB")
else:
    print("\nNo matching notebooks found in the project directory.")
    print("The notebook may be stored outside this directory.")



Searching for Feature_Engineering notebooks...

Found: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\notebooks\Feature_Engineering.ipynb
Size: 182.8 KB


In [39]:

from pathlib import Path
import json

notebook_path = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
    r"\notebooks\Feature_Engineering.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "causal_data.csv",
    "display",
    "mailer",
    "promotion_features",
    "read_csv",
    "groupby"
]

print("ORIGINAL PROMOTION DATA PROCESSING")
print("=" * 60)

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matched = [word for word in keywords if word.lower() in source.lower()]

    if matched:
        print(f"\n--- CELL {i} | Keywords: {', '.join(matched)} ---")
        print(source[:6000])



ORIGINAL PROMOTION DATA PROCESSING

--- CELL 0 | Keywords: read_csv ---
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current sessi

In [40]:
from pathlib import Path
import json

notebook_path = (
    Path(r"D:\Econocausal Double machine learning project")
    / "EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
    / "notebooks"
    / "Feature_Engineering.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "causal_data.csv",
    "display_code_count",
    "mailer_code_count",
    "display",
    "mailer",
    "promotions.groupby",
]

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(k.lower() in source.lower() for k in keywords):
        print(f"\n{'=' * 65}")
        print(f"CELL {i}")
        print("=" * 65)
        print(source)


CELL 1

import os
import pandas as pd

base_path = (
    "/kaggle/input/datasets/frtgnn/"
    "dunnhumby-the-complete-journey/"
)

for filename in ["transaction_data.csv", "causal_data.csv"]:
    path = os.path.join(base_path, filename)

    df = pd.read_csv(path, nrows=5)

    print(f"\n{'=' * 45}")
    print(filename)
    print("Columns:", df.columns.tolist())
    display(df.head())


CELL 2

import pandas as pd
import numpy as np

base_path = "/kaggle/input/datasets/frtgnn/dunnhumby-the-complete-journey/"

transactions = pd.read_csv(base_path + "transaction_data.csv")
promotions = pd.read_csv(base_path + "causal_data.csv")

print("Transactions shape:", transactions.shape)
print("Promotions shape:", promotions.shape)



CELL 3

def inspect_data(df, name):
    print(f"\n{'=' * 50}")
    print(name)
    print(f"{'=' * 50}")

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    display(df.isnull().sum().to_frame("missing_count"))

    print("\nExact dupli

In [41]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m", "kaggle",
        "datasets", "files",
        "frtgnn/dunnhumby-the-complete-journey"
    ],
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", result.returncode)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Error:")
    print(result.stderr)



Return code: 0
name                       size  creationDate                
--------------------  ---------  --------------------------  
campaign_desc.csv           540  2019-11-07 06:59:03.908000  
campaign_table.csv        95874  2019-11-07 06:59:03.928000  
causal_data.csv       695858427  2019-11-07 06:59:25.548000  
coupon.csv              2822804  2019-11-07 06:59:12.723000  
coupon_redempt.csv        54108  2019-11-07 06:59:12.642000  
hh_demographic.csv        44349  2019-11-07 06:59:12.522000  
product.csv             6429896  2019-11-07 06:59:12.584000  
transaction_data.csv  141742346  2019-11-07 06:59:14.969000  



In [42]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m", "kaggle",
        "datasets", "download",
        "frtgnn/dunnhumby-the-complete-journey",
        "-f", "causal_data.csv",
        "--help"
    ],
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout[:4000])
print(result.stderr[:1000])



usage: __main__.py datasets download [-h] [-f FILE_NAME] [-p PATH] [-w]
                                     [--unzip] [-o] [-q]
                                     [dataset]

options:
  -h, --help            show this help message and exit
  dataset               Dataset URL suffix in format <owner>/<dataset-name> (use "kaggle datasets list" to show options)
  -f FILE_NAME, --file FILE_NAME
                        File name, all files downloaded if not provided
                        (use "kaggle datasets files -d <dataset>" to show options)
  -p PATH, --path PATH  Folder where file(s) will be downloaded, defaults to current working directory
  -w, --wp              Download files to current working path
  --unzip               Unzip the downloaded file. Will delete the zip file when completed.
  -o, --force           Skip check whether local version of file is up to date, force file download
  -q, --quiet           Suppress printing information about the upload/download progress




In [43]:

import subprocess
import sys
from pathlib import Path

# Use a temporary directory outside the project
temp_dir = Path(r"D:\EconoCausal_temp")
temp_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m", "kaggle",
    "datasets", "download",
    "frtgnn/dunnhumby-the-complete-journey",
    "-f", "causal_data.csv",
    "-p", str(temp_dir),
    "-q"
]

print("Downloading only causal_data.csv...")
print("Temporary location:", temp_dir)

result = subprocess.run(cmd, text=True, capture_output=True)

if result.returncode != 0:
    print("Download failed:")
    print(result.stderr[-3000:])
else:
    print("Download command completed.")
    for file in temp_dir.iterdir():
        print(file.name, f"{file.stat().st_size / (1024**2):.2f} MB")



Temporary location: D:\EconoCausal_temp
Download command completed.
causal_data.csv.zip 98.31 MB


In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path(r"D:\EconoCausal_temp\causal_data.csv")

# Read only the columns we need, using efficient data types
usecols = ["PRODUCT_ID", "STORE_ID", "WEEK_NO", "display", "mailer"]

chunksize = 500_000
parts = []

for i, chunk in enumerate(
    pd.read_csv(
        csv_path,
        usecols=usecols,
        dtype={
            "PRODUCT_ID": "string",
            "STORE_ID": "string",
            "WEEK_NO": "Int16",
            "display": "category",
            "mailer": "category",
        },
        chunksize=chunksize,
    ),
    start=1,
):
    display_code = chunk["display"].astype("string").str.strip()
    mailer_code = chunk["mailer"].astype("string").str.strip()

    chunk["display_exposed"] = (
        display_code.ne("0") & display_code.notna()
    ).astype("int8")

    chunk["mailer_exposed"] = (
        mailer_code.ne("0") & mailer_code.notna()
    ).astype("int8")

    chunk["any_promotion"] = (
        (chunk["display_exposed"] == 1)
        | (chunk["mailer_exposed"] == 1)
    ).astype("int8")

    keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

    # Keep the compact treatment data for aggregation
    parts.append(
        chunk.groupby(keys, observed=True, sort=False)[
            ["display_exposed", "mailer_exposed", "any_promotion"]
        ].max().reset_index()
    )

    print(f"Processed chunk {i}: {len(chunk):,} rows")

Processed chunk 1: 500,000 rows
Processed chunk 2: 500,000 rows
Processed chunk 3: 500,000 rows
Processed chunk 4: 500,000 rows
Processed chunk 5: 500,000 rows
Processed chunk 6: 500,000 rows
Processed chunk 7: 500,000 rows
Processed chunk 8: 500,000 rows
Processed chunk 9: 500,000 rows
Processed chunk 10: 500,000 rows
Processed chunk 11: 500,000 rows
Processed chunk 12: 500,000 rows
Processed chunk 13: 500,000 rows
Processed chunk 14: 500,000 rows
Processed chunk 15: 500,000 rows
Processed chunk 16: 500,000 rows
Processed chunk 17: 500,000 rows
Processed chunk 18: 500,000 rows
Processed chunk 19: 500,000 rows
Processed chunk 20: 500,000 rows
Processed chunk 21: 500,000 rows
Processed chunk 22: 500,000 rows
Processed chunk 23: 500,000 rows
Processed chunk 24: 500,000 rows
Processed chunk 25: 500,000 rows
Processed chunk 26: 500,000 rows
Processed chunk 27: 500,000 rows
Processed chunk 28: 500,000 rows
Processed chunk 29: 500,000 rows
Processed chunk 30: 500,000 rows
Processed chunk 31:

In [2]:
import pandas as pd

keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
treatment_cols = [
    "display_exposed",
    "mailer_exposed",
    "any_promotion"
]

print("Combining processed chunks...")

treatment_features = pd.concat(parts, ignore_index=True)

print("Rows before final aggregation:", len(treatment_features))

# Count how many partial records exist for each key
key_counts = (
    treatment_features.groupby(keys, observed=True, sort=False)
    .size()
)

print("Keys appearing in multiple chunks:", (key_counts > 1).sum())

# Combine partial results at product-store-week level
treatment_features = (
    treatment_features.groupby(
        keys, observed=True, sort=False, as_index=False
    )[treatment_cols]
    .max()
)

print("\nFinal treatment dataset shape:", treatment_features.shape)
print("Duplicate keys:", treatment_features.duplicated(keys).sum())

print("\nTreatment summaries:")
for col in treatment_cols:
    print(f"\n{col}")
    print(treatment_features[col].value_counts(dropna=False).sort_index())

display(treatment_features.head())

Combining processed chunks...
Rows before final aggregation: 36771279
Keys appearing in multiple chunks: 0

Final treatment dataset shape: (36771279, 6)
Duplicate keys: 0

Treatment summaries:

display_exposed
display_exposed
0    21023537
1    15747742
Name: count, dtype: int64

mailer_exposed
mailer_exposed
0    11534087
1    25237192
Name: count, dtype: int64

any_promotion
any_promotion
1    36771279
Name: count, dtype: int64


,PRODUCT_ID,STORE_ID,WEEK_NO,display_exposed,mailer_exposed,any_promotion
0,26190,286,70,0,1,1
1,26190,288,70,0,1,1
2,26190,289,70,0,1,1
3,26190,292,70,0,1,1
4,26190,293,70,0,1,1


In [4]:
import pandas as pd

keys = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
treatment_cols = [
    "display_exposed",
    "mailer_exposed",
    "any_promotion"
]

print("Combining existing processed chunks...")

treatment_features = pd.concat(parts, ignore_index=True)

# Combine partial results at product-store-week level
treatment_features = (
    treatment_features
    .groupby(keys, observed=True, sort=False, as_index=False)
    [treatment_cols]
    .max()
)

print("\nFinal treatment dataset shape:", treatment_features.shape)

print("\nTreatment summaries:")
for col in treatment_cols:
    print(f"\n{col}:")
    print(treatment_features[col].value_counts(dropna=False).sort_index())

print("\nDuplicate keys:",
      treatment_features.duplicated(keys).sum())

display(treatment_features.head())

Combining existing processed chunks...

Final treatment dataset shape: (36771279, 6)

Treatment summaries:

display_exposed:
display_exposed
0    21023537
1    15747742
Name: count, dtype: int64

mailer_exposed:
mailer_exposed
0    11534087
1    25237192
Name: count, dtype: int64

any_promotion:
any_promotion
1    36771279
Name: count, dtype: int64

Duplicate keys: 0


,PRODUCT_ID,STORE_ID,WEEK_NO,display_exposed,mailer_exposed,any_promotion
0,26190,286,70,0,1,1
1,26190,288,70,0,1,1
2,26190,289,70,0,1,1
3,26190,292,70,0,1,1
4,26190,293,70,0,1,1


In [5]:
from pathlib import Path

# Recalculate combined treatment from the two exposure indicators
treatment_features["any_promotion"] = (
    (treatment_features["display_exposed"] == 1)
    | (treatment_features["mailer_exposed"] == 1)
).astype("int8")

# Validate
print("Treatment summary:")
for col in ["display_exposed", "mailer_exposed", "any_promotion"]:
    print(f"\n{col}:")
    print(treatment_features[col].value_counts(dropna=False).sort_index())

# Save into the existing project folder
project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

output_path = project_dir / "promotion_treatment_features.csv"
treatment_features.to_csv(output_path, index=False)

print("\nSaved successfully:", output_path)
print("File size (MB):", round(output_path.stat().st_size / 1024**2, 2))

Treatment summary:

display_exposed:
display_exposed
0    21023537
1    15747742
Name: count, dtype: int64

mailer_exposed:
mailer_exposed
0    11534087
1    25237192
Name: count, dtype: int64

any_promotion:
any_promotion
1    36771279
Name: count, dtype: int64

Saved successfully: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\promotion_treatment_features.csv
File size (MB): 768.55


In [6]:
# Check the actual data types and values
print("Data types:")
print(treatment_features[
    ["display_exposed", "mailer_exposed", "any_promotion"]
].dtypes)

print("\nExposure combinations:")
print(
    treatment_features.groupby(
        ["display_exposed", "mailer_exposed"],
        observed=True
    ).size()
)

print("\nRows where both indicators are zero:")
print(
    (
        (treatment_features["display_exposed"] == 0)
        & (treatment_features["mailer_exposed"] == 0)
    ).sum()
)

Data types:
display_exposed    int8
mailer_exposed     int8
any_promotion      int8
dtype: object

Exposure combinations:
display_exposed  mailer_exposed
0                1                 21023537
1                0                 11534087
                 1                  4213655
dtype: int64

Rows where both indicators are zero:
0


In [10]:

from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

print("Searching project folder...")

matches = list(project_dir.rglob("transaction_weekly_final.csv"))

if matches:
    print("\nFound transaction dataset:")
    for path in matches:
        print(path)
else:
    print("\nNot found in project folder.")
    print("Searching D drive...")

    matches = list(Path("D:/").rglob("transaction_weekly_final.csv"))

    if matches:
        for path in matches[:20]:
            print(path)
    else:
        print("File not found on D drive.")



Searching project folder...

Found transaction dataset:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\transaction_weekly_final.csv
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\Output\causal output\Output\transaction_weekly_final.csv


In [1]:
from pathlib import Path
import duckdb

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

transaction_path = (
    project_dir / "refutation test" / "transaction_weekly_final.csv"
)

treatment_path = project_dir / "promotion_treatment_features.csv"
output_path = project_dir / "refutation test" / "transactions_with_treatments.parquet"

con = duckdb.connect()

# Join on disk using DuckDB
con.execute("""
COPY (
    SELECT
        t.*,
        p.display_exposed,
        p.mailer_exposed,
        p.any_promotion
    FROM read_csv_auto(?, sample_size=200000) AS t
    LEFT JOIN read_csv_auto(?, sample_size=200000) AS p
      ON CAST(t.PRODUCT_ID AS VARCHAR) = CAST(p.PRODUCT_ID AS VARCHAR)
     AND CAST(t.STORE_ID AS VARCHAR) = CAST(p.STORE_ID AS VARCHAR)
     AND CAST(t.WEEK_NO AS VARCHAR) = CAST(p.WEEK_NO AS VARCHAR)
) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
""", [str(transaction_path), str(treatment_path), str(output_path)])

print("Join completed!")
print("Saved to:", output_path)
print("Output size (MB):", round(output_path.stat().st_size / 1024**2, 2))

Join completed!
Saved to: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\transactions_with_treatments.parquet
Output size (MB): 13.83


In [2]:
import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

joined_path = (
    project_dir / "refutation test" / "transactions_with_treatments.parquet"
)

con = duckdb.connect()

# 1. Check row counts and treatment coverage
print("JOINED DATASET SUMMARY")

summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(display_exposed) AS display_matched,
        COUNT(mailer_exposed) AS mailer_matched,
        COUNT(any_promotion) AS promotion_matched
    FROM read_parquet(?)
""", [str(joined_path)]).df()

display(summary)

# 2. Check treatment variation
print("\nTREATMENT DISTRIBUTIONS")

distribution = con.execute("""
    SELECT
        display_exposed,
        mailer_exposed,
        any_promotion,
        COUNT(*) AS observations
    FROM read_parquet(?)
    GROUP BY ALL
    ORDER BY ALL
""", [str(joined_path)]).df()

display(distribution)

# 3. Check outcome columns
print("\nAVAILABLE COLUMNS")
print(
    con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(joined_path)]
    ).df()["column_name"].tolist()
)

JOINED DATASET SUMMARY


,total_rows,display_matched,mailer_matched,promotion_matched
0,2370784,483320,483320,483320



TREATMENT DISTRIBUTIONS


,display_exposed,mailer_exposed,any_promotion,observations
0,0,1,1,246268
1,1,0,1,133132
2,1,1,1,103920
3,<NA>,<NA>,<NA>,1887464



AVAILABLE COLUMNS
['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'TOTAL_QUANTITY', 'TOTAL_SALES', 'TOTAL_RETAIL_DISC', 'TOTAL_COUPON_DISC', 'TOTAL_COUPON_MATCH_DISC', 'TRANSACTION_ROWS', 'UNIQUE_BASKETS', 'ZERO_QUANTITY_ROWS', 'ZERO_SALES_ROWS', 'EXTREME_QUANTITY_ROWS', 'PRICE_PROXY', 'ZERO_OR_NEGATIVE_QUANTITY_FLAG', 'ZERO_SALES_FLAG', 'MISSING_PRICE_PROXY_FLAG', 'EXTREME_QUANTITY_FLAG', 'NONPOSITIVE_PRICE_PROXY_FLAG', 'display_exposed', 'mailer_exposed', 'any_promotion']


In [4]:

import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

joined_path = (
    project_dir / "refutation test" / "transactions_with_treatments.parquet"
)

con = duckdb.connect()

# Compare sales for matched observations by display exposure
result = con.execute("""
    SELECT
        CAST(display_exposed AS INTEGER) AS treatment,
        COUNT(*) AS observations,
        AVG(CAST(TOTAL_SALES AS DOUBLE)) AS mean_sales,
        MEDIAN(CAST(TOTAL_SALES AS DOUBLE)) AS median_sales,
        STDDEV_SAMP(CAST(TOTAL_SALES AS DOUBLE)) AS sales_std
    FROM read_parquet(?)
    WHERE display_exposed IS NOT NULL
      AND TOTAL_SALES IS NOT NULL
    GROUP BY CAST(display_exposed AS INTEGER)
    ORDER BY treatment
""", [str(joined_path)]).df()

display(result)

# Calculate the unadjusted difference in means
if len(result) == 2:
    difference = (
        result.loc[result["treatment"] == 1, "mean_sales"].iloc[0]
        - result.loc[result["treatment"] == 0, "mean_sales"].iloc[0]
    )
    print("\nUnadjusted difference in mean sales:", round(difference, 4))
    print("Note: This is a descriptive comparison, not a causal estimate.")
else:
    print("Both treatment groups are required for comparison.")



,treatment,observations,mean_sales,median_sales,sales_std
0,0,246268,3.383697,2.205,3.940647
1,1,237052,3.056100,2.000,3.498595



Unadjusted difference in mean sales: -0.3276
Note: This is a descriptive comparison, not a causal estimate.


In [5]:
import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

joined_path = (
    project_dir / "refutation test" / "transactions_with_treatments.parquet"
)

con = duckdb.connect()

# Inspect column names and data types
schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(joined_path)]
).df()

print("AVAILABLE VARIABLES")
print(schema[["column_name", "column_type"]].to_string(index=False))

AVAILABLE VARIABLES
                   column_name column_type
                    PRODUCT_ID      BIGINT
                      STORE_ID      BIGINT
                       WEEK_NO      BIGINT
                TOTAL_QUANTITY      BIGINT
                   TOTAL_SALES      DOUBLE
             TOTAL_RETAIL_DISC      DOUBLE
             TOTAL_COUPON_DISC      DOUBLE
       TOTAL_COUPON_MATCH_DISC      DOUBLE
              TRANSACTION_ROWS      BIGINT
                UNIQUE_BASKETS      BIGINT
            ZERO_QUANTITY_ROWS      BIGINT
               ZERO_SALES_ROWS      BIGINT
         EXTREME_QUANTITY_ROWS      BIGINT
                   PRICE_PROXY      DOUBLE
ZERO_OR_NEGATIVE_QUANTITY_FLAG      BIGINT
               ZERO_SALES_FLAG      BIGINT
      MISSING_PRICE_PROXY_FLAG      BIGINT
         EXTREME_QUANTITY_FLAG      BIGINT
  NONPOSITIVE_PRICE_PROXY_FLAG      BIGINT
               display_exposed      BIGINT
                mailer_exposed      BIGINT
                 any_promotion    

In [6]:

import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

joined_path = (
    project_dir / "refutation test" / "transactions_with_treatments.parquet"
)

output_path = (
    project_dir / "refutation test" / "dml_analysis_features.parquet"
)

con = duckdb.connect()

# Create lagged sales features using only earlier weeks
con.execute("""
COPY (
    SELECT
        *,
        LAG(TOTAL_SALES, 1) OVER (
            PARTITION BY PRODUCT_ID, STORE_ID
            ORDER BY WEEK_NO
        ) AS previous_week_sales,
        LAG(TOTAL_QUANTITY, 1) OVER (
            PARTITION BY PRODUCT_ID, STORE_ID
            ORDER BY WEEK_NO
        ) AS previous_week_quantity,
        AVG(TOTAL_SALES) OVER (
            PARTITION BY PRODUCT_ID, STORE_ID
            ORDER BY WEEK_NO
            ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING
        ) AS previous_4_week_avg_sales
    FROM read_parquet(?)
) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
""", [str(joined_path), str(output_path)])

print("Historical sales features created.")
print("Saved to:", output_path)
print("Output size (MB):", round(output_path.stat().st_size / 1024**2, 2))



Historical sales features created.
Saved to: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\dml_analysis_features.parquet
Output size (MB): 22.29


In [7]:
import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

features_path = (
    project_dir / "refutation test" / "dml_analysis_features.parquet"
)

con = duckdb.connect()

# Check treatment coverage and availability of historical features
summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE display_exposed IS NOT NULL
        ) AS rows_with_display_treatment,

        COUNT(*) FILTER (
            WHERE display_exposed = 0
        ) AS display_unexposed,

        COUNT(*) FILTER (
            WHERE display_exposed = 1
        ) AS display_exposed_rows,

        COUNT(*) FILTER (
            WHERE previous_week_sales IS NOT NULL
        ) AS rows_with_previous_week_sales,

        COUNT(*) FILTER (
            WHERE previous_4_week_avg_sales IS NOT NULL
        ) AS rows_with_four_week_history,

        COUNT(*) FILTER (
            WHERE TOTAL_SALES IS NULL
        ) AS missing_sales
    FROM read_parquet(?)
""", [str(features_path)]).df()

display(summary)

,total_rows,rows_with_display_treatment,display_unexposed,display_exposed_rows,rows_with_previous_week_sales,rows_with_four_week_history,missing_sales
0,2370784,483320,246268,237052,1430713,1430713,0


In [8]:
import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

features_path = (
    project_dir / "refutation test" / "dml_analysis_features.parquet"
)

sample_path = (
    project_dir / "refutation test" / "dml_analysis_sample.parquet"
)

con = duckdb.connect()

# Keep matched observations with valid outcomes and historical features.
# Do not treat unmatched promotion records as untreated.
con.execute("""
COPY (
    SELECT
        PRODUCT_ID,
        STORE_ID,
        WEEK_NO,
        CAST(TOTAL_SALES AS DOUBLE) AS outcome,
        CAST(display_exposed AS INTEGER) AS treatment,
        CAST(mailer_exposed AS INTEGER) AS mailer_exposed,
        CAST(previous_week_sales AS DOUBLE) AS previous_week_sales,
        CAST(previous_week_quantity AS DOUBLE) AS previous_week_quantity,
        CAST(previous_4_week_avg_sales AS DOUBLE) AS previous_4_week_avg_sales
    FROM read_parquet(?)
    WHERE display_exposed IS NOT NULL
      AND TOTAL_SALES IS NOT NULL
      AND previous_week_sales IS NOT NULL
      AND previous_week_quantity IS NOT NULL
      AND previous_4_week_avg_sales IS NOT NULL
) TO ? (FORMAT PARQUET, COMPRESSION ZSTD)
""", [str(features_path), str(sample_path)])

# Summarize the resulting sample
summary = con.execute("""
    SELECT
        COUNT(*) AS sample_size,
        COUNT(*) FILTER (WHERE treatment = 0) AS untreated,
        COUNT(*) FILTER (WHERE treatment = 1) AS treated,
        AVG(outcome) AS mean_sales,
        COUNT(*) FILTER (
            WHERE previous_week_sales IS NULL
               OR previous_week_quantity IS NULL
               OR previous_4_week_avg_sales IS NULL
        ) AS rows_missing_history
    FROM read_parquet(?)
""", [str(sample_path)]).df()

display(summary)

print("\nSaved analysis sample:", sample_path)
print("File size (MB):", round(sample_path.stat().st_size / 1024**2, 2))

,sample_size,untreated,treated,mean_sales,rows_missing_history
0,331066,167551,163515,3.249832,0



Saved analysis sample: D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\dml_analysis_sample.parquet
File size (MB): 2.44


In [9]:

import duckdb
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

sample_path = (
    project_dir / "refutation test" / "dml_analysis_sample.parquet"
)

# Load the compact sample only
con = duckdb.connect()
df = con.execute(
    "SELECT * FROM read_parquet(?)",
    [str(sample_path)]
).df()

# Outcome, treatment, and covariates
Y = df["outcome"].astype(float).to_numpy()
D = df["treatment"].astype(float).to_numpy()

X = df[
    [
        "mailer_exposed",
        "previous_week_sales",
        "previous_week_quantity",
        "previous_4_week_avg_sales",
    ]
].astype(float)

# Cross-fitting
cv = KFold(n_splits=5, shuffle=True, random_state=42)

model_y = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42
)

model_d = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42
)

# Out-of-fold predictions
y_hat = cross_val_predict(model_y, X, Y, cv=cv, n_jobs=1)

d_hat = cross_val_predict(
    model_d, X, D.astype(int), cv=cv, method="predict_proba", n_jobs=1
)[:, 1]

# Residualize outcome and treatment
y_resid = Y - y_hat
d_resid = D - d_hat

# Estimate the final-stage treatment coefficient
theta = np.sum(d_resid * y_resid) / np.sum(d_resid ** 2)

# Approximate standard error for the orthogonal-score estimate
scores = d_resid * (y_resid - theta * d_resid)
se = np.sqrt(np.mean(scores ** 2) / np.mean(d_resid ** 2) ** 2 / len(D))

ci_low = theta - 1.96 * se
ci_high = theta + 1.96 * se

print("PRELIMINARY CROSS-FITTED DML RESULTS")
print("------------------------------------")
print("Sample size:", len(df))
print("Estimated treatment effect:", round(theta, 6))
print("Approximate standard error:", round(se, 6))
print("Approximate 95% confidence interval:",
      (round(ci_low, 6), round(ci_high, 6)))
print("Outcome RMSE:", round(np.sqrt(mean_squared_error(Y, y_hat)), 6))
print("Mean predicted treatment propensity:", round(float(np.mean(d_hat)), 6))
print("Propensity score range:",
      (round(float(np.min(d_hat)), 4), round(float(np.max(d_hat)), 4)))



PRELIMINARY CROSS-FITTED DML RESULTS
------------------------------------
Sample size: 331066
Estimated treatment effect: 0.065846
Approximate standard error: 0.012992
Approximate 95% confidence interval: (np.float64(0.040383), np.float64(0.09131))
Outcome RMSE: 2.775349
Mean predicted treatment propensity: 0.494119
Propensity score range: (0.001, 1.0)


In [10]:

import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

sample_path = (
    project_dir / "refutation test" / "dml_analysis_sample.parquet"
)

con = duckdb.connect()

result = con.execute("""
SELECT
    treatment,
    COUNT(*) AS n,
    AVG(outcome) AS mean_sales,
    AVG(previous_week_sales) AS mean_previous_sales,
    AVG(previous_week_quantity) AS mean_previous_quantity,
    AVG(previous_4_week_avg_sales) AS mean_previous_4wk_sales,
    AVG(mailer_exposed) AS mailer_rate
FROM read_parquet(?)
GROUP BY treatment
ORDER BY treatment
""", [str(sample_path)]).df()

display(result)

,treatment,n,mean_sales,mean_previous_sales,mean_previous_quantity,mean_previous_4wk_sales,mailer_rate
0,0,167551,3.434606,3.488219,1.607457,3.487774,1.000000
1,1,163515,3.060497,3.063177,1.618500,3.051201,0.455212


In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Reuse df, Y, X and D from the previous DML cell if they still exist.
# If not, reload the compact sample.
if "df" not in globals() or len(df) == 0:
    import duckdb
    from pathlib import Path

    project_dir = Path(
        r"D:\Econocausal Double machine learning project"
        r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
    )
    sample_path = project_dir / "refutation test" / "dml_analysis_sample.parquet"
    df = duckdb.connect().execute(
        "SELECT * FROM read_parquet(?)", [str(sample_path)]
    ).df()

Y = df["outcome"].to_numpy(dtype=float)
D = df["treatment"].to_numpy(dtype=int)

X = df[
    [
        "mailer_exposed",
        "previous_week_sales",
        "previous_week_quantity",
        "previous_4_week_avg_sales",
    ]
].astype(float)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

y_hat = cross_val_predict(
    RandomForestRegressor(
        n_estimators=80, min_samples_leaf=20,
        n_jobs=-1, random_state=42
    ),
    X, Y, cv=cv, n_jobs=1
)

d_hat = cross_val_predict(
    RandomForestClassifier(
        n_estimators=80, min_samples_leaf=20,
        n_jobs=-1, random_state=42
    ),
    X, D, cv=cv, method="predict_proba", n_jobs=1
)[:, 1]

print("Propensity percentiles:",
      np.percentile(d_hat, [0, 1, 5, 50, 95, 99, 100]))

# Restrict to observations with reasonable estimated propensity scores.
mask = (d_hat >= 0.05) & (d_hat <= 0.95)

yr = Y[mask] - y_hat[mask]
dr = D[mask] - d_hat[mask]

theta = np.sum(dr * yr) / np.sum(dr ** 2)
scores = dr * (yr - theta * dr)
se = np.sqrt(np.mean(scores ** 2) / np.mean(dr ** 2) ** 2 / len(dr))

print("\nOVERLAP SENSITIVITY CHECK")
print("Retained observations:", int(mask.sum()), "of", len(mask))
print("Estimate:", round(theta, 6))
print("Approx. 95% CI:",
      (round(theta - 1.96 * se, 6), round(theta + 1.96 * se, 6)))

Propensity percentiles: [3.67647059e-04 4.40506266e-02 1.10777597e-01 3.62454267e-01
 1.00000000e+00 1.00000000e+00 1.00000000e+00]

OVERLAP SENSITIVITY CHECK
Retained observations: 237836 of 331066
Estimate: 0.067304
Approx. 95% CI: (np.float64(0.041823), np.float64(0.092785))


In [12]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

output_path = (
    project_dir / "refutation test" / "dml_preliminary_results.csv"
)

results = pd.DataFrame([
    ["Original analysis sample", 331066, "observations"],
    ["Treated observations", 163515, "display exposed"],
    ["Untreated observations", 167551, "display unexposed"],
    ["Initial DML estimate", 0.065846, "sales units"],
    ["Initial approximate standard error", 0.012992, "sales units"],
    ["Initial approximate 95% CI lower", 0.040383, "sales units"],
    ["Initial approximate 95% CI upper", 0.091310, "sales units"],
    ["Overlap sensitivity sample", 237836, "observations"],
    ["Trimmed DML estimate", 0.067304, "sales units"],
    ["Trimmed approximate 95% CI lower", 0.041823, "sales units"],
    ["Trimmed approximate 95% CI upper", 0.092785, "sales units"],
    ["Transaction promotion match rate", 20.39, "percent"],
], columns=["metric", "value", "unit"])

results.to_csv(output_path, index=False)

print("Results saved successfully:")
print(output_path)
display(results)

Results saved successfully:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\dml_preliminary_results.csv


,metric,value,unit
0,Original analysis sample,331066.000000,observations
1,Treated observations,163515.000000,display exposed
2,Untreated observations,167551.000000,display unexposed
3,Initial DML estimate,0.065846,sales units
4,Initial approximate standard error,0.012992,sales units
5,Initial approximate 95% CI lower,0.040383,sales units
6,Initial approximate 95% CI upper,0.091310,sales units
7,Overlap sensitivity sample,237836.000000,observations
8,Trimmed DML estimate,0.067304,sales units
9,Trimmed approximate 95% CI lower,0.041823,sales units


In [13]:

import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

joined_path = (
    project_dir
    / "refutation test"
    / "transactions_with_treatments.parquet"
)

con = duckdb.connect()

# 1. Overall match coverage
print("OVERALL COVERAGE")

overall = con.execute("""
    SELECT
        COUNT(*) AS transaction_rows,
        COUNT(*) FILTER (
            WHERE display_exposed IS NOT NULL
        ) AS matched_rows,
        COUNT(*) FILTER (
            WHERE display_exposed IS NULL
        ) AS unmatched_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE display_exposed IS NOT NULL
            ) / COUNT(*), 2
        ) AS match_rate_percent
    FROM read_parquet(?)
""", [str(joined_path)]).df()

display(overall)

# 2. Coverage by week
print("\nCOVERAGE BY WEEK")

weekly = con.execute("""
    SELECT
        WEEK_NO,
        COUNT(*) AS transaction_rows,
        COUNT(*) FILTER (
            WHERE display_exposed IS NOT NULL
        ) AS matched_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE display_exposed IS NOT NULL
            ) / COUNT(*), 2
        ) AS match_rate_percent
    FROM read_parquet(?)
    GROUP BY WEEK_NO
    ORDER BY WEEK_NO
""", [str(joined_path)]).df()

display(weekly)

# 3. Summarize weekly match-rate variation
print("\nWEEKLY MATCH-RATE SUMMARY")

print(
    weekly["match_rate_percent"].describe(
        percentiles=[0.25, 0.50, 0.75]
    ).round(2)
)



OVERALL COVERAGE


,transaction_rows,matched_rows,unmatched_rows,match_rate_percent
0,2370784,483320,1887464,20.39



COVERAGE BY WEEK


,WEEK_NO,transaction_rows,matched_rows,match_rate_percent
0,1,1855,0,0.00
1,2,3597,0,0.00
2,3,4630,0,0.00
3,4,5175,0,0.00
4,5,6866,0,0.00
...,...,...,...,...
97,98,28377,5932,20.90
98,99,29365,6528,22.23
99,100,26682,5196,19.47
100,101,24440,4506,18.44



WEEKLY MATCH-RATE SUMMARY
count    102.00
mean      19.16
std        6.35
min        0.00
25%       19.16
50%       20.42
75%       21.78
max       27.46
Name: match_rate_percent, dtype: float64


In [14]:

import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

joined_path = (
    project_dir / "refutation test" / "transactions_with_treatments.parquet"
)

con = duckdb.connect()

result = con.execute("""
    SELECT
        COUNT(*) AS transaction_rows_weeks_9_101,

        COUNT(*) FILTER (
            WHERE display_exposed IS NOT NULL
        ) AS matched_rows,

        COUNT(*) FILTER (
            WHERE display_exposed IS NULL
        ) AS unmatched_rows,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE display_exposed IS NOT NULL
            ) / COUNT(*),
            2
        ) AS match_rate_percent,

        COUNT(DISTINCT PRODUCT_ID) AS products,
        COUNT(DISTINCT STORE_ID) AS stores,
        COUNT(DISTINCT WEEK_NO) AS weeks

    FROM read_parquet(?)
    WHERE WEEK_NO BETWEEN 9 AND 101
""", [str(joined_path)]).df()

display(result)



,transaction_rows_weeks_9_101,matched_rows,unmatched_rows,match_rate_percent,products,stores,weeks
0,2300718,483320,1817398,21.01,90334,576,93


In [2]:

import duckdb
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

test_dir = project_dir / "refutation test"
sample_path = test_dir / "dml_analysis_sample.parquet"
results_path = test_dir / "dml_preliminary_results.csv"
report_path = test_dir / "EconoCausal_Final_Status.txt"

# Validate the existing compact analysis sample
con = duckdb.connect()
summary = con.execute("""
    SELECT
        COUNT(*) AS sample_size,
        COUNT(*) FILTER (WHERE treatment = 0) AS untreated,
        COUNT(*) FILTER (WHERE treatment = 1) AS treated,
        COUNT(*) FILTER (WHERE outcome IS NULL) AS missing_outcomes,
        COUNT(*) FILTER (
            WHERE previous_week_sales IS NULL
               OR previous_week_quantity IS NULL
               OR previous_4_week_avg_sales IS NULL
        ) AS missing_history
    FROM read_parquet(?)
""", [str(sample_path)]).df().iloc[0]

report = f"""
ECONOCAUSAL — PROJECT STATUS
============================

DATA PREPARATION
- Treatment dataset constructed from original display/mailer codes.
- Transaction and promotion records joined.
- Historical sales features created.
- Compact analysis sample saved.

FINAL SAMPLE VALIDATION
- Sample size: {int(summary['sample_size']):,}
- Display-unexposed: {int(summary['untreated']):,}
- Display-exposed: {int(summary['treated']):,}
- Missing outcomes: {int(summary['missing_outcomes']):,}
- Missing historical features: {int(summary['missing_history']):,}

PRELIMINARY DML RESULTS
- Initial estimate: +0.065846 sales units.
- Initial approximate 95% CI: [0.040383, 0.091310].
- Trimmed estimate: +0.067304 sales units.
- Trimmed approximate 95% CI: [0.041823, 0.092785].

IMPORTANT LIMITATIONS
- Only 20.39% of all transaction keys matched promotion records.
- Restricting to weeks 9–101 increased coverage only to 21.01%.
- Unmatched transactions must not automatically be treated as untreated.
- Propensity scores showed extreme values.
- Mailer exposure was imbalanced between display groups.
- The current estimates are preliminary, not validated causal effects.
- Formal DoWhy refutation tests have not yet been completed.
- Unobserved confounding has not been ruled out.

REMAINING WORK
1. Resolve promotion-to-transaction key coverage.
2. Validate pre-treatment covariates, treatment overlap, and DML inference.
3. Run appropriate DoWhy refutation tests and document the results.
"""

report_path.write_text(report.strip(), encoding="utf-8")

print("Final sample validation:")
display(summary.to_frame("value"))

print("\nProject status saved to:")
print(report_path)



Final sample validation:


,value
sample_size,331066
untreated,167551
treated,163515
missing_outcomes,0
missing_history,0



Project status saved to:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\EconoCausal_Final_Status.txt


In [3]:
import duckdb
import pandas as pd
from pathlib import Path

# Check DoWhy installation
try:
    from dowhy import CausalModel
except ImportError:
    print("DoWhy is not installed in the current environment.")
    print("Run %pip install dowhy in a separate cell, then rerun this cell.")
    raise

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

sample_path = (
    project_dir / "refutation test" / "dml_analysis_sample.parquet"
)

results_dir = project_dir / "refutation test"
results_dir.mkdir(parents=True, exist_ok=True)

# Load a reproducible sample from the compact Parquet file
con = duckdb.connect()

data = con.execute("""
    SELECT
        outcome,
        treatment,
        mailer_exposed,
        previous_week_sales,
        previous_week_quantity,
        previous_4_week_avg_sales
    FROM read_parquet(?)
    USING SAMPLE 25000 ROWS (reservoir, 42)
""", [str(sample_path)]).df()

print("Refutation sample size:", len(data))
print("Treatment distribution:")
print(data["treatment"].value_counts())

# Define the causal model
model = CausalModel(
    data=data,
    treatment="treatment",
    outcome="outcome",
    common_causes=[
        "mailer_exposed",
        "previous_week_sales",
        "previous_week_quantity",
        "previous_4_week_avg_sales",
    ],
)

identified_estimand = model.identify_effect(
    proceed_when_unidentifiable=True
)

estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    test_significance=True,
)

print("\nDoWhy preliminary estimate:")
print(estimate)

# Run two refutation tests
refutation_results = []

for method in [
    "placebo_treatment_refuter",
    "random_common_cause",
]:
    print(f"\nRunning refutation: {method}")

    try:
        refutation = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name=method,
        )

        print(refutation)
        refutation_results.append({
            "test": method,
            "result": str(refutation),
            "status": "completed",
        })

    except Exception as exc:
        print("Test failed:", str(exc))
        refutation_results.append({
            "test": method,
            "result": str(exc),
            "status": "failed",
        })

# Save the refutation log
refutation_path = results_dir / "dowhy_refutation_results.csv"

pd.DataFrame(refutation_results).to_csv(
    refutation_path,
    index=False,
)

print("\nRefutation log saved to:")
print(refutation_path)

d:\EconoCausal_Env\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Refutation sample size: 25000
Treatment distribution:
treatment
0    12534
1    12466
Name: count, dtype: int64

DoWhy preliminary estimate:
*** Causal Estimate ***

## Identified estimand
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                                                                         ↪
────────────(E[outcome|mailer_exposed,previous_week_sales,previous_4_week_avg_ ↪
d[treatment]                                                                   ↪

↪                               
↪ sales,previous_week_quantity])
↪                               
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→outcome then P(outcome|treatment,mailer_exposed,previous_week_sales,previous_4_week_avg_sales,previous_week_quantity,U) = P(outcome|treatment,mailer_exposed,previous_week_sales,previous_4_week_avg_sales,previous_week_quantity)

## Realized estimand
b: outcome~treatment+mailer_exposed+previous_w

In [4]:

from pathlib import Path
import pandas as pd

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

results_dir = project_dir / "refutation test"
results_dir.mkdir(parents=True, exist_ok=True)

# Consolidated preliminary findings
results = pd.DataFrame([
    ["Initial cross-fitted DML estimate", 0.065846, "Sales units"],
    ["Initial approximate 95% CI lower", 0.040383, "Sales units"],
    ["Initial approximate 95% CI upper", 0.091310, "Sales units"],
    ["Trimmed DML estimate", 0.067304, "Sales units"],
    ["Trimmed approximate 95% CI lower", 0.041823, "Sales units"],
    ["Trimmed approximate 95% CI upper", 0.092785, "Sales units"],
    ["DML analysis sample", 331066, "Observations"],
    ["Propensity-trimmed sample", 237836, "Observations"],
    ["Transaction-promotion match rate", 20.39, "Percent"],
    ["DoWhy sample", 25000, "Observations"],
    ["DoWhy linear estimate", -0.023620, "Sales units"],
    ["DoWhy estimate p-value", 0.6276, "P-value"],
    ["Placebo refutation p-value", 0.88, "P-value"],
    ["Random common cause refutation p-value", 0.98, "P-value"],
], columns=["metric", "value", "unit"])

results_path = results_dir / "EconoCausal_Consolidated_Results.csv"
results.to_csv(results_path, index=False)

report_path = results_dir / "EconoCausal_Final_Analysis_Summary.txt"

report = """
ECONOCAUSAL: DYNAMIC PRICING VIA DOUBLE MACHINE LEARNING
PRELIMINARY ANALYSIS SUMMARY
========================================================

COMPLETED
- Constructed display and mailer exposure indicators.
- Joined promotion features with weekly transaction outcomes.
- Created historical sales features.
- Prepared a 331,066-observation analysis sample.
- Calculated preliminary cross-fitted DML-style estimates.
- Performed propensity-trimming sensitivity analysis.
- Ran DoWhy linear-regression estimation on a 25,000-row sample.
- Completed placebo-treatment and random-common-cause refutation tests.
- Saved consolidated results.

PRELIMINARY DML
- Initial estimate: +0.065846 sales units.
- Approximate 95% interval: [0.040383, 0.091310].
- Trimmed estimate: +0.067304 sales units.
- Approximate 95% interval: [0.041823, 0.092785].

DOWHY SUBSAMPLE
- Estimate: -0.023620 sales units.
- P-value: 0.6276.
- Placebo refutation p-value: 0.88.
- Random common cause refutation p-value: 0.98.

IMPORTANT LIMITATIONS
- Only 20.39% of transaction keys matched promotion records.
- Unmatched transactions were not established to be untreated.
- Propensity scores had extreme values.
- Mailer exposure was imbalanced between display groups.
- The DML and DoWhy estimates differed in sign and came from different
  estimators and samples.
- The refutation tests do not prove that unobserved confounding is absent.
- Covariate timing and causal identification assumptions still require review.

CONCLUSION
The data-processing and preliminary estimation pipeline is operational.
The evidence is not yet sufficient to claim a validated causal effect of
display placement on sales. Further work should resolve key coverage,
validate treatment and pre-treatment covariates, and reconcile the estimator
results before a final causal conclusion is reported.
"""

report_path.write_text(report.strip(), encoding="utf-8")

print("Consolidated results saved:")
print(results_path)

print("\nAnalysis summary saved:")
print(report_path)

print("\nConsolidated results:")
display(results)



Consolidated results saved:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\EconoCausal_Consolidated_Results.csv

Analysis summary saved:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\EconoCausal_Final_Analysis_Summary.txt

Consolidated results:


,metric,value,unit
0,Initial cross-fitted DML estimate,0.065846,Sales units
1,Initial approximate 95% CI lower,0.040383,Sales units
2,Initial approximate 95% CI upper,0.091310,Sales units
3,Trimmed DML estimate,0.067304,Sales units
4,Trimmed approximate 95% CI lower,0.041823,Sales units
5,Trimmed approximate 95% CI upper,0.092785,Sales units
6,DML analysis sample,331066.000000,Observations
7,Propensity-trimmed sample,237836.000000,Observations
8,Transaction-promotion match rate,20.390000,Percent
9,DoWhy sample,25000.000000,Observations


In [5]:

import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

transaction_path = (
    project_dir / "refutation test" / "transaction_weekly_final.csv"
)
treatment_path = project_dir / "promotion_treatment_features.csv"

con = duckdb.connect()

# Keep DuckDB from using excessive memory or multiple threads.
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")

# Compare distinct product/store/week keys using DuckDB.
# The output is only one summary row.
query = """
WITH tx AS (
    SELECT DISTINCT
        CAST(PRODUCT_ID AS VARCHAR) AS product,
        CAST(STORE_ID AS VARCHAR) AS store,
        CAST(WEEK_NO AS INTEGER) AS week
    FROM read_csv_auto(?, sample_size=10000)
    WHERE WEEK_NO BETWEEN 9 AND 101
),
pr AS (
    SELECT DISTINCT
        CAST(PRODUCT_ID AS VARCHAR) AS product,
        CAST(STORE_ID AS VARCHAR) AS store,
        CAST(WEEK_NO AS INTEGER) AS week
    FROM read_csv_auto(?, sample_size=10000)
)
SELECT
    (SELECT COUNT(*) FROM tx) AS transaction_keys,
    (SELECT COUNT(*) FROM pr) AS promotion_keys,
    (SELECT COUNT(*) FROM tx
        WHERE product IN (SELECT product FROM pr)
    ) AS transaction_keys_with_product_overlap,
    (SELECT COUNT(*) FROM tx
        WHERE store IN (SELECT store FROM pr)
    ) AS transaction_keys_with_store_overlap,
    (SELECT COUNT(*) FROM tx
        WHERE week IN (SELECT week FROM pr)
    ) AS transaction_keys_with_week_overlap
"""

result = con.execute(
    query,
    [str(transaction_path), str(treatment_path)]
).df()

display(result)



,transaction_keys,promotion_keys,transaction_keys_with_product_overlap,transaction_keys_with_store_overlap,transaction_keys_with_week_overlap
0,2300718,36771279,2083313,2265861,2300718


In [1]:
import duckdb
from pathlib import Path

project_dir = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

sample_path = (
    project_dir / "refutation test" / "dml_analysis_sample.parquet"
)

con = duckdb.connect(":memory:")
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")

result = con.execute("""
    SELECT
        COUNT(*) AS sample_size,
        COUNT(DISTINCT PRODUCT_ID) AS products,
        COUNT(DISTINCT STORE_ID) AS stores,
        COUNT(DISTINCT WEEK_NO) AS weeks,
        COUNT(*) FILTER (WHERE treatment = 0) AS display_unexposed,
        COUNT(*) FILTER (WHERE treatment = 1) AS display_exposed,
        COUNT(*) FILTER (WHERE mailer_exposed = 1) AS mailer_exposed
    FROM read_parquet(?)
""", [str(sample_path)]).df()

display(result)

,sample_size,products,stores,weeks,display_unexposed,display_exposed,mailer_exposed
0,331066,12603,114,93,167551,163515,241985


In [2]:

from pathlib import Path
import pandas as pd
import duckdb

# ============================================================
# ECONOCAUSAL: DYNAMIC PRICING VIA DOUBLE MACHINE LEARNING
# Final analysis consolidation
# ============================================================

PROJECT_DIR = Path(
    r"D:\Econocausal Double machine learning project"
    r"\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning"
)

RESULTS_DIR = PROJECT_DIR / "refutation test"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATH = RESULTS_DIR / "dml_analysis_sample.parquet"
REFUTATION_PATH = RESULTS_DIR / "dowhy_refutation_results.csv"

RESULTS_CSV = RESULTS_DIR / "EconoCausal_Final_Results.csv"
REPORT_PATH = RESULTS_DIR / "EconoCausal_Project_Conclusion.txt"

# ------------------------------------------------------------
# 1. Validate the existing analysis sample
#    Reads the compact Parquet file; no raw CSV processing.
# ------------------------------------------------------------

con = duckdb.connect(":memory:")
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")

sample = con.execute("""
    SELECT
        COUNT(*) AS sample_size,
        COUNT(*) FILTER (WHERE treatment = 0) AS display_unexposed,
        COUNT(*) FILTER (WHERE treatment = 1) AS display_exposed,
        COUNT(*) FILTER (WHERE outcome IS NULL) AS missing_outcomes,
        COUNT(*) FILTER (
            WHERE previous_week_sales IS NULL
               OR previous_week_quantity IS NULL
               OR previous_4_week_avg_sales IS NULL
        ) AS missing_history
    FROM read_parquet(?)
""", [str(SAMPLE_PATH)]).df().iloc[0]

# ------------------------------------------------------------
# 2. Consolidate results already obtained
#    Do not rerun any models.
# ------------------------------------------------------------

results = [
    ("Analysis sample size", int(sample["sample_size"]), "observations"),
    ("Display-unexposed observations", int(sample["display_unexposed"]), "observations"),
    ("Display-exposed observations", int(sample["display_exposed"]), "observations"),
    ("Missing outcomes", int(sample["missing_outcomes"]), "observations"),
    ("Missing historical features", int(sample["missing_history"]), "observations"),
    ("Initial DML estimate", 0.065846, "sales units"),
    ("Initial DML approximate CI lower", 0.040383, "sales units"),
    ("Initial DML approximate CI upper", 0.091310, "sales units"),
    ("Trimmed DML estimate", 0.067304, "sales units"),
    ("Trimmed DML approximate CI lower", 0.041823, "sales units"),
    ("Trimmed DML approximate CI upper", 0.092785, "sales units"),
    ("Transaction-promotion match rate", 20.39, "percent"),
    ("Weeks 9-101 match rate", 21.01, "percent"),
    ("DoWhy sample size", 25000, "observations"),
    ("DoWhy linear-regression estimate", -0.023620, "sales units"),
    ("DoWhy estimate p-value", 0.6276, "p-value"),
    ("Placebo refutation p-value", 0.88, "p-value"),
    ("Random common cause refutation p-value", 0.98, "p-value"),
]

results_df = pd.DataFrame(
    results,
    columns=["metric", "value", "unit"]
)

results_df.to_csv(RESULTS_CSV, index=False)

# ------------------------------------------------------------
# 3. Read the refutation log if available
# ------------------------------------------------------------

if REFUTATION_PATH.exists():
    refutations = pd.read_csv(REFUTATION_PATH)
    refutation_status = (
        refutations[["test", "status"]].to_string(index=False)
        if {"test", "status"}.issubset(refutations.columns)
        else refutations.to_string(index=False)
    )
else:
    refutation_status = "Refutation log not found."

# ------------------------------------------------------------
# 4. Write the final project conclusion
# ------------------------------------------------------------

report = f"""
ECONOCAUSAL: DYNAMIC PRICING VIA DOUBLE MACHINE LEARNING
PROJECT CONCLUSION AND ANALYSIS STATUS
========================================================

COMPLETED WORK
--------------
1. Accessed the original Dunnhumby promotion data.
2. Constructed display and mailer exposure indicators.
3. Aggregated promotion records at product/store/week level.
4. Joined promotion information with weekly transaction data.
5. Created historical sales and quantity features.
6. Prepared the preliminary DML analysis sample.
7. Calculated an initial cross-fitted DML-style estimate.
8. Performed a propensity-trimming sensitivity analysis.
9. Ran a separate DoWhy linear-regression analysis on 25,000 rows.
10. Executed placebo-treatment and random-common-cause refutation tests.
11. Saved the numerical results and refutation log.

CURRENT SAMPLE
--------------
Sample size: {int(sample['sample_size']):,}
Display-unexposed: {int(sample['display_unexposed']):,}
Display-exposed: {int(sample['display_exposed']):,}
Missing outcomes: {int(sample['missing_outcomes']):,}
Missing historical features: {int(sample['missing_history']):,}

PRELIMINARY ESTIMATES
---------------------
Initial DML estimate: +0.065846 sales units.
Initial approximate 95% confidence interval: [0.040383, 0.091310].

Propensity-trimmed DML estimate: +0.067304 sales units.
Trimmed approximate 95% confidence interval: [0.041823, 0.092785].

DoWhy linear-regression estimate: -0.023620 sales units.
DoWhy estimate p-value: 0.6276.

REFUTATION TESTS
----------------
Placebo-treatment refutation p-value: 0.88.
Random-common-cause refutation p-value: 0.98.

Refutation execution log:
{refutation_status}

LIMITATIONS
-----------
- Only about 20.39% of all transaction keys matched promotion records.
- Restricting to weeks 9-101 increased coverage only to about 21.01%.
- Unmatched transaction records cannot automatically be classified as untreated.
- The propensity scores showed extreme values.
- Mailer exposure was imbalanced between display groups.
- The preliminary DML and DoWhy estimates have opposite signs.
- The estimators use different specifications and samples; their estimates
  are not directly interchangeable.
- Treatment timing and pre-treatment covariate validity require further review.
- Refutation tests do not prove that unobserved confounding is absent.

FINAL CONCLUSION
----------------
The data-preparation pipeline and preliminary estimation workflow are
operational, and results have been saved for review.

The current evidence is insufficient to establish a validated causal effect
of display placement on sales. The positive DML estimates should be treated
as preliminary rather than conclusive causal findings.

The next substantive research task is to resolve the promotion-key coverage
problem, validate treatment and covariate timing, examine overlap, and
reconcile the estimators before making a causal claim.

This report records the work completed and the remaining limitations.
It does not certify that all causal assumptions have been satisfied.
"""

REPORT_PATH.write_text(report.strip(), encoding="utf-8")

print("=" * 58)
print("ECONOCAUSAL PROJECT SUMMARY SAVED")
print("=" * 58)

print("\nResults CSV:")
print(RESULTS_CSV)

print("\nConclusion report:")
print(REPORT_PATH)

print("\nFinal sample validation:")
display(sample.to_frame("value"))

print("\nConsolidated results:")
display(results_df)

con.close()


ECONOCAUSAL PROJECT SUMMARY SAVED

Results CSV:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\EconoCausal_Final_Results.csv

Conclusion report:
D:\Econocausal Double machine learning project\EconoCausal-Dynamic-Pricing-via-Double-Machine-Learning\refutation test\EconoCausal_Project_Conclusion.txt

Final sample validation:


,value
sample_size,331066
display_unexposed,167551
display_exposed,163515
missing_outcomes,0
missing_history,0



Consolidated results:


,metric,value,unit
0,Analysis sample size,331066.000000,observations
1,Display-unexposed observations,167551.000000,observations
2,Display-exposed observations,163515.000000,observations
3,Missing outcomes,0.000000,observations
4,Missing historical features,0.000000,observations
5,Initial DML estimate,0.065846,sales units
6,Initial DML approximate CI lower,0.040383,sales units
7,Initial DML approximate CI upper,0.091310,sales units
8,Trimmed DML estimate,0.067304,sales units
9,Trimmed DML approximate CI lower,0.041823,sales units


# Conclusion

The EconoCausal project aimed to investigate the relationship between promotional activities and product sales using **Double Machine Learning (DML)** and causal inference techniques. Using the Dunnhumby – The Complete Journey dataset, the analysis focused on evaluating the effect of in-store display exposure on sales while accounting for historical sales patterns and other relevant promotional features.

The final analysis sample contained **331,066 observations**, comprising 163,515 display-exposed observations and 167,551 display-unexposed observations. The sample had no missing outcome values or missing historical features, providing a complete dataset for the subsequent analysis.

The preliminary Double Machine Learning analysis estimated a positive effect of display exposure on sales of approximately **0.0658 sales units**. After trimming observations with extreme estimated treatment propensities, the estimate remained positive at approximately **0.0673 sales units**. The consistency between these two estimates suggests that the preliminary result was relatively stable under this particular sensitivity check. However, the confidence intervals reported by the analysis are approximate and should not be interpreted as definitive statistical evidence without further validation.

The project also used the DoWhy framework to estimate the effect on a separate sample of 25,000 observations. This analysis produced an estimated effect of approximately **−0.0236 sales units**, with a p-value of **0.628**. The placebo-treatment and random-common-cause refutation tests produced p-values of **0.88** and **0.98**, respectively. These tests did not identify significant changes under the tested refutations, but they do not establish that the causal estimate is correct or that all potential sources of bias have been eliminated. Furthermore, the DoWhy estimate differed in direction from the preliminary DML estimates, indicating that the findings are not yet consistent across analytical approaches.

An important limitation was identified during data preparation: the transaction-to-promotion match rate was approximately **20.39%**, increasing only to **21.01%** when transaction records were restricted to weeks 9–101. This limited overlap may affect the representativeness of the analysis sample. In addition, the relationship between display and mailer exposure requires further investigation before the results can be interpreted as the isolated causal effect of display promotions.

Overall, the project demonstrates the implementation of a data-driven causal analysis workflow, incorporating large-scale data preparation, treatment-feature engineering, historical feature creation, Double Machine Learning estimation, propensity-based sensitivity analysis, and DoWhy refutation testing. The findings provide an initial indication of a possible relationship between display exposure and sales, but **the causal effect has not yet been conclusively established**.

Future work should prioritize investigating the transaction-to-promotion key mismatch, validating treatment definitions and comparison groups, confirming that all adjustment variables are measured before treatment, and applying an appropriate inference procedure that accounts for dependence between repeated product, store, and week observations. Further validation using consistent treatment definitions and comparable estimation samples is also necessary to reconcile the differing DML and DoWhy results.

In conclusion, EconoCausal establishes a preliminary technical foundation for evaluating promotional effectiveness using causal machine learning. With further data-quality checks and statistical validation, this framework could support more reliable promotional analysis and inform future data-driven pricing and retail decision-making.
